# 74 — End-to-end: SASRec recall + wRRF union -> LGBM rerank -> nDCG, + Blind-A submission

One notebook for the whole improved pipeline.

Stage 1 (recall): content-fused SASRec as a 4th channel in wrrf_union_v1. Reports recall@{20,100} for union vs union+SASRec (the G1 result; +0.0586 @100 on full dev).

Stage 2 (rerank train): build LGBM LambdaRank features WITH the SASRec channel (sasrec_rank_inv feature) and train two models — with and without that feature — reporting held-out val nDCG@20 (the G2 number + its control).

Stage 3 (end-to-end DEV nDCG): union+SASRec recall@100 -> LGBM rerank -> nDCG@20 on dev (we have golds here). Compares recall-only vs LGBM(no sasrec feat) vs LGBM(+sasrec feat).

Stage 4 (Blind-A submission, separate logic at the end): run the full pipeline (config 191 = union+SASRec -> LGBM(+sasrec) -> v5-kto responder) over the 80 Blind-A queries and package prediction.json. NOTE: Blind-A has no public golds — nDCG there is scored by the CodaBench server, not locally. Stage 3 is the local nDCG signal.

Run order: cells top to bottom. Stages 1-3 are fast; Stage 4 is the long responder run (~35-65 min) — run it only when Stage 3 confirms the lift.

In [ ]:
# 1) Setup. Disable JAX GPU preallocation BEFORE any import pulls JAX in
# (datasets/transformers import JAX transitively; it grabs ~75% VRAM on first use).
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
os.environ.setdefault('TF_FORCE_GPU_ALLOW_GROWTH', 'true')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'recall-union-lgbm'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

# Symlink the persistent caches from Drive. retrieval_v2 holds sasrec/, lgbm/,
# ctx_cache/; dense holds the Qwen query/catalog cache for dense_metadata_qwen3.
DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
for name, drive_subdir in [
    ('retrieval_v2', 'recsys2026_retrieval_v2_cache'),
    ('dense', 'recsys2026_dense_cache'),
]:
    src = f'{DRIVE_BASE}/{drive_subdir}'
    dst = f'{LOCAL_BASE}/{name}'
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst): os.unlink(dst)
    elif os.path.exists(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)

# Deps: retrieval stack + lightgbm (Stages 1-3) + responder/inference (Stage 4).
!pip install -q --upgrade 'transformers>=4.40' 'accelerate>=0.30' 'peft>=0.11'     'datasets' 'pandas<3.0' 'tqdm' 'huggingface_hub' 'sentence-transformers>=3.0'     'FlagEmbedding>=1.3' 'bm25s' 'lightgbm' 'scikit-learn'     'omegaconf' 'pyyaml' 'trl>=0.12.0' 'torchao>=0.17'


## Stage 1 — recall: content-fused SASRec as the 4th union channel

In [ ]:
# 3) Ensure the content-fused SASRec checkpoint exists (sasrec_v1). Trains it
# only if missing (it persists on the Drive cache across runtimes).
import os, sys
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
CACHE_DIR = '/content/recsys2026/experiments/cache'
ITEM_DB = 'talkpl-ai/TalkPlayData-Challenge-Track-Metadata'
CORPUS = ['track_name', 'artist_name', 'album_name']
SASREC_CKPT = f'{CACHE_DIR}/retrieval_v2/sasrec/sasrec_v1/sasrec.pt'
if os.path.exists(SASREC_CKPT):
    print('[sasrec] checkpoint present, skipping train:', SASREC_CKPT)
else:
    print('[sasrec] training content-fused SASRec (sasrec_v1, ~10 epochs)...')
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out sasrec_v1 --epochs 10


In [ ]:
# 4) Build the FULL dev eval set + report recall@{20,100} for union vs union+SASRec.
# A2 FIX (2026-05-30): query now appends listener_goal, matching the production
# crs_baseline query. Blind-A nDCG 0.29 >> old dev 0.16 was largely this mismatch;
# the dev harness was pessimistic. Downstream stages reuse this query.
# Defines the shared dev variables reused by Stage 3.
import numpy as np
import pandas as pd
from datasets import load_dataset
from mcrs.db_item.music_catalog import MusicCatalogDB
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)
dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

queries, golds, user_ids, played, user_dialogs = [], [], [], [], []
goal_categories, goal_specificities, turn_numbers = [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    goal_txt = (goal.get('listener_goal') or '').strip()  # A2: prod query includes this
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        _q = chr(10).join(lines)
        if goal_txt:
            _q = _q + chr(10) + 'goal: ' + goal_txt  # A2: train/serve parity w/ crs_baseline
        queries.append(_q)
        user_dialogs.append(build_user_dialog(prior.to_dict('records')))
        golds.append(music['content'])
        user_ids.append(sess.get('user_id'))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        goal_categories.append(goal.get('category'))
        goal_specificities.append(goal.get('specificity'))
        turn_numbers.append(tn)
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
print('[dev] built', len(queries), 'turns')

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))

base = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={})
sas = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                            CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
cb = base.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs = sas.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print('=== Stage 1 recall (FULL dev, n=' + str(len(golds)) + ') ===')
print('  union (3-chan) : @20=' + str(round(recall_at(cb, 20), 4)) + ' @100=' + str(round(recall_at(cb, 100), 4)))
print('  union + SASRec : @20=' + str(round(recall_at(cs, 20), 4)) + ' @100=' + str(round(recall_at(cs, 100), 4)))
print('  delta @100     :', round(recall_at(cs, 100) - recall_at(cb, 100), 4))


In [ ]:
# 4-strat) Turn-stratified nDCG@20 reporting (Tier-0 #1).
# Blind-A is single-turn / zero-history, so the turn-1 stratum is the ONLY honest
# proxy for Blind; the flat mean over all 8 turns over-credits the played-track-
# seeded channels (same_artist / SASRec sequence / related_artist) that are dead
# on Blind. Use report_by_turn(ranked) anywhere you'd call ndcg20(ranked), and
# gate any ship-to-Blind decision on the turn1 number, NOT overall.
# Requires cell 4 (defines golds, turn_numbers).
from mcrs.eval_ndcg import ndcg_by_turn, format_by_turn

def report_by_turn(ranked, label=''):
    rep = ndcg_by_turn(ranked, golds, turn_numbers, k=20)
    if label:
        print(f'== {label} ==')
    print(format_by_turn(rep))
    return rep

# Example: report_by_turn([c[:20] for c in fused], 'wrrf fused (baseline)')


In [ ]:
# 4-strat-recall) Turn-1 / WALL recall@20 instrument (plan Phase 0). The binding
# metric for Blind (100% turn-1, ~99% new-artist wall) is TURN-1 recall@20 over
# wall golds, NOT overall recall@100. Reports per-channel + union turn-1 recall@20
# and the wall subset, and (optional toggle) re-gates the cold content channels
# (attributes/lyrics) at the right cutoff. Requires cell 4 (queries, golds,
# turn_numbers, played, user_ids, ctx, cs, sas, item_db). No GPU/API.
import numpy as np
from mcrs.eval_ndcg import recall_by_turn

def _artist_of(tid):
    md = item_db.metadata_dict.get(tid) or {}
    a = md.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

is_wall = np.array([_artist_of(g) not in {_artist_of(t) for t in p}
                    for g, p in zip(golds, played)])
turn1 = np.array([t == 1 for t in turn_numbers])
print(f'[strat] turns={len(golds)} turn1={int(turn1.sum())} wall={int(is_wall.sum())} '
      f'turn1&wall={int((turn1 & is_wall).sum())}')

def _recall_mask(cands, k, mask):
    idx = np.where(mask)[0]
    if len(idx) == 0: return float('nan')
    return float(np.mean([1.0 if golds[i] in cands[i][:k] else 0.0 for i in idx]))

def strat_recall(cands, label=''):
    rep = recall_by_turn(cands, golds, turn_numbers, k=20)
    t1 = rep.get('turn1'); w1 = _recall_mask(cands, 20, turn1 & is_wall)
    print(f'  {label:30s} recall@20 overall={rep["overall"]:.4f} '
          f'turn1={(t1 if t1 is not None else float("nan")):.4f} turn1&wall={w1:.4f}')
    return rep

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
print('=== per-channel recall@20 (turn-1 vs wall; session channels ~0 at turn-1) ===')
for lab, ps in zip(labels, per_sub):
    strat_recall(ps, lab)
print('=== union ===')
strat_recall(cs, 'union+SASRec (cs)')

# Optional: re-gate the cold content channels on turn-1 wall recall@20 (free no-LLM
# levers if they lift it). Heavy-ish (builds dense channels); off by default.
CHECK_CONTENT_CHANNELS = False
if CHECK_CONTENT_CHANNELS:
    from mcrs.retrieval_modules import load_retrieval_module
    for flag in ('use_attributes', 'use_lyrics'):
        u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                                  CACHE_DIR, extra_config={flag: True})
        cu = u.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids,
                                            batch_context=ctx)
        strat_recall(cu, f'union+{flag}')


In [ ]:
# 4-attr) Tier-1 #3.1a VALIDATION (no retrain): does attributes-qwen3 as a 2nd
# dense channel lift TURN-1 recall@100 (the cold/Blind proxy)? Recall is leak-
# free, so we gate here BEFORE any feature rebuild + reranker retrain. Proceed to
# rebuild->retrain->nDCG only if turn-1 recall lifts beyond inter-seed noise.
# Requires cell 4 (cs, queries, golds, turn_numbers, ctx, user_ids).
from mcrs.eval_ndcg import recall_by_turn, format_by_turn

print(format_by_turn(recall_by_turn(cs, golds, turn_numbers, k=100),
                     'union+SASRec (baseline)'))
for _w in (0.3, 0.4, 0.7):
    _u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                               CACHE_DIR, extra_config={'use_sasrec': True,
                               'w_sasrec': 1.0, 'use_attributes': True,
                               'w_attributes': _w})
    _cu = _u.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids,
                                          batch_context=ctx)
    print(format_by_turn(recall_by_turn(_cu, golds, turn_numbers, k=100),
                         f'+attributes w={_w}'))


In [ ]:
# 4-enr) Tier-1 #3.2 VALIDATION (no retrain): does adding culture + profile to
# the retrieval query (raw_enriched) lift TURN-1 recall@100? Builds base vs
# enriched queries via build_retrieval_query (the SAME fn serve uses), retrieves
# with the 3-channel union, reports recall_by_turn. Gate: wire serve + feature-
# builder + retrain only if turn-1 recall lifts. Requires cell 4 (base, golds,
# turn_numbers, ctx, user_ids, item_db, dev) + 4-strat (recall_by_turn helpers).
from mcrs.crs_baseline import build_retrieval_query
from mcrs.eval_ndcg import recall_by_turn, format_by_turn

q_base, q_enr = [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    goal = sess.get('conversation_goal') or {}
    gt = (goal.get('listener_goal') or '').strip()
    up = sess.get('user_profile') or {}
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        sm = [{'role': ('assistant' if t['role'] == 'music' else t['role']),
               'content': (item_db.id_to_metadata(t['content'])
                           if t['role'] == 'music' else t['content'])}
              for _, t in prior.iterrows()]
        q_base.append(build_retrieval_query(sm, mode='raw_with_goal', goal_text=gt))
        q_enr.append(build_retrieval_query(sm, mode='raw_enriched', goal_text=gt,
                                           user_profile=up))

cbb = base.batch_text_to_item_retrieval(q_base, topk=100, user_ids=user_ids, batch_context=ctx)
cee = base.batch_text_to_item_retrieval(q_enr,  topk=100, user_ids=user_ids, batch_context=ctx)
print(format_by_turn(recall_by_turn(cbb, golds, turn_numbers, 100), 'raw_with_goal (baseline)'))
print(format_by_turn(recall_by_turn(cee, golds, turn_numbers, 100), 'raw_enriched (culture+profile)'))


In [ ]:
# 4-cf) Tier-1 #3.4 VALIDATION (no retrain): does the cf-bpr user x item channel
# lift TURN-1 recall@100? cf_bpr is cold-firable (query-independent, uses user_id;
# fires for WARM users, empty for cold -> 0 RRF contribution, no regression). The
# channel + use_cfbpr gate already exist; this only validates. Gate: enable +
# retrain only if turn-1 recall lifts. Requires cell 4 (cb, queries, golds,
# turn_numbers, ctx, user_ids) + 4-strat (recall_by_turn helpers).
from mcrs.eval_ndcg import recall_by_turn, format_by_turn

# Warm-user coverage bounds the achievable lift.
_ucf = load_retrieval_module('cf_bpr', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR)
_warm = sum(1 for u in user_ids if u in _ucf.user_embs)
print(f'[cf-bpr] warm dev turns: {_warm}/{len(user_ids)} ({_warm/len(user_ids):.1%})')
print(format_by_turn(recall_by_turn(cb, golds, turn_numbers, 100), 'union 3-chan (baseline)'))
for _w in (0.25, 0.5, 1.0):
    _u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                               CACHE_DIR, extra_config={'use_cfbpr': True, 'w_cfbpr': _w})
    _cu = _u.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    print(format_by_turn(recall_by_turn(_cu, golds, turn_numbers, 100), f'+cf_bpr w={_w}'))


In [ ]:
# 4-sq) Tier-1 #3.1b VALIDATION (no retrain): does the LLM structured-query
# channel lift TURN-1 recall@100? HEAVY: loads Qwen2.5-7B and extracts a content-
# only query per dev turn (cached by query hash, so re-runs are fast). Gate:
# offline-cache the 122k train turns + feature rebuild + retrain ONLY if turn-1
# recall lifts. Requires cell 4 (cb, queries, golds, turn_numbers, ctx, user_ids)
# + 4-strat (recall_by_turn helpers).
from mcrs.eval_ndcg import recall_by_turn, format_by_turn

_usq = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                             CACHE_DIR, extra_config={'use_structured_query': True,
                             'w_structured_query': 0.5})
_csq = _usq.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids,
                                         batch_context=ctx)
print(format_by_turn(recall_by_turn(cb, golds, turn_numbers, 100), 'union 3-chan (baseline)'))
print(format_by_turn(recall_by_turn(_csq, golds, turn_numbers, 100), '+structured_query w=0.5'))


In [ ]:
# 4-tt) Tier-1 #3.3 VALIDATION (no reranker retrain): does the two-tower content
# channel reach the new-artist WALL? Reports the SAME metric set as 4-pg so the two
# wall-channels are directly comparable: OVERALL recall@{20,100}, nDCG@20, and
# WALL-gold rescue (golds union+SASRec MISSED that the two-tower now reaches).
# Gate on OVERALL dev (Blind is turn-UNIFORM, 2026-06-08); per-turn kept as a
# diagnostic. A/B is ON TOP of the shipped union+SASRec stack (cs), not bare.
# Requires the trained checkpoint (run 4-tt-train) + cell 4 (cs, queries, golds,
# turn_numbers, ctx, user_ids) + 4-strat (recall_by_turn/format_by_turn).
import numpy as np
from mcrs.eval_ndcg import recall_by_turn, format_by_turn, ndcg_at_k

def _hit(cands, k=100):
    return [1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]
def _ndcg20(cands):
    return float(np.mean([ndcg_at_k(g, c, 20) for c, g in zip(cands, golds)]))

cs_hit = _hit(cs)  # union+SASRec = the shipped recall stack (baseline)
print(f'baseline union+SASRec : recall@20={recall_at(cs,20):.4f} '
      f'recall@100={np.mean(cs_hit):.4f} nDCG@20={_ndcg20(cs):.4f}')
for _w in (0.3, 0.5, 0.7):
    _u = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
                               CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0,
                               'use_two_tower': True, 'w_two_tower': _w})
    _ctt = _u.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids,
                                            batch_context=ctx)
    tt_hit = _hit(_ctt)
    rescued = [1.0 if (cs_hit[i] == 0 and tt_hit[i] == 1) else 0.0 for i in range(len(golds))]
    print(f'+two_tower w={_w}      : recall@20={recall_at(_ctt,20):.4f} '
          f'recall@100={np.mean(tt_hit):.4f} nDCG@20={_ndcg20(_ctt):.4f}  '
          f'WALL rescued={np.mean(rescued):.4f} ({int(np.sum(rescued))}/{len(golds)})')

# Per-turn diagnostic at the best weight is informative but NOT the gate.
print(format_by_turn(recall_by_turn(cs, golds, turn_numbers, 100), 'union+SASRec (by-turn)'))
print('READ: gate on OVERALL recall@100 + nDCG@20 lift and WALL rescue >~0.02 (same bar as 4-pg).')
print('      recall lifts but nDCG flat -> recall surfaced golds the reranker cannot yet')
print('      convert; rebuild features + retrain (12a-style) before the final judgement.')


In [ ]:
# 4-tt-train) Tier-1 #3.3: TRAIN the two-tower (Colab GPU). Smoke first with
# --n-sessions 300 --epochs 1, then full. Time-based holdout (leak-free). Writes
# {CACHE_DIR}/retrieval_v2/two_tower/two_tower_v1/two_tower.pt. Requires cell 1/3
# (CACHE_DIR, ITEM_DB). Run ONCE; the 4-tt validation cell loads the result.
!cd /content/recsys2026 && python -u scripts/train_two_tower.py \
    --cache-dir {CACHE_DIR} --dataset-name {ITEM_DB} --out two_tower_v1 \
    --n-sessions 300 --epochs 1   # smoke; drop --n-sessions + raise --epochs for full


In [ ]:
# 4-pg) propose-ground vs the 42% WALL — GEMINI backend (L4-safe; the 7B OOMs on L4).
# PURE RECALL diagnostic (NO reranker): does Gemini propose-ground reach golds that the
# union+SASRec (config-194) pool MISSES? Reports grounding yield + WALL-gold rescue.
# Requires cell 4 (cs, queries, golds, ctx, user_ids). Gemini = 1 API call/query, so
# subset to N_PG; proposals cache to disk -> re-runs are fast.
# (For Blackwell + local 7B instead, set pg_model='Qwen/Qwen2.5-7B-Instruct'.)
import os, numpy as np
!pip install -q google-genai
try:
    from google.colab import userdata
    os.environ.setdefault('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY'))
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY'), 'set GEMINI_API_KEY (Colab secret) before running'

N_PG = 2000                                   # subset (full 8000 ~ 4x cost/time)
q, g, c, u, base_cands = queries[:N_PG], golds[:N_PG], ctx[:N_PG], user_ids[:N_PG], cs[:N_PG]

pg_ret = load_retrieval_module('propose_ground', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'pg_model': 'gemini-2.5-flash', 'n_proposals': 20,
                  'batch_size': 4, 'inner_dense': 'dense_metadata_qwen3_instruct'})
pg = pg_ret.batch_text_to_item_retrieval(q, topk=100, user_ids=u, batch_context=c)

def _hit(cands, gs, k=100):
    return [1.0 if gg in cc[:k] else 0.0 for cc, gg in zip(cands, gs)]

yield_n  = np.mean([len(x) for x in pg])
nonempty = np.mean([1.0 if x else 0.0 for x in pg])
pg_hit, cs_hit = _hit(pg, g), _hit(base_cands, g)
union_hit = [max(a, b) for a, b in zip(pg_hit, cs_hit)]
rescued   = [1.0 if (cs_hit[i] == 0 and pg_hit[i] == 1) else 0.0 for i in range(len(g))]

print(f'[pg] subset N={N_PG} | yield avg {yield_n:.1f}/q, {nonempty:.1%} non-empty')
print(f'[pg] standalone recall@100 : {np.mean(pg_hit):.4f}')
print(f'[pg] baseline union+SASRec : {np.mean(cs_hit):.4f}')
print(f'[pg] cs UNION pg recall@100: {np.mean(union_hit):.4f}  (upper bound if folded in)')
print(f'[pg] WALL golds RESCUED    : {np.mean(rescued):.4f}  ({int(np.sum(rescued))}/{len(g)})')
print('READ: rescue > ~0.02 + yield >~50% non-empty -> real wall break, fold in + retrain. Else drop.')


In [ ]:
# 4-pg-conv) TRACK C (plan step 3): does propose-ground + LLM-rank convert the
# new-artist WALL? Add pg's world-knowledge new-artist golds to the pool (recall^),
# then let the dev-validated LLM listwise ranker lift them into top-20 (conversion^).
# Operates on the TURN-1 subset (the Blind proxy). Reports pg turn-1 WALL recall@
# 20/@100, then turn-1 nDCG@20 of LLM-rank on cs vs on union(cs,pg). The decision:
# does LLM(cs+pg) beat LLM(cs)=0.2256 ? Requires cell 4 (queries, golds, turn_numbers,
# played, ctx, user_ids, cs, item_db) + GEMINI key. ~$0.6-0.9 (pg=flash, rerank=flash-lite).
import os, numpy as np, pandas as pd
from datasets import load_dataset
from mcrs.eval_ndcg import ndcg_by_turn
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.rerankers.llm_listwise_rerank import LLMListwiseReranker
try:
    from google.colab import userdata
    os.environ.setdefault('GEMINI_API_KEY', userdata.get('GEMINI_API_KEY'))
except Exception:
    pass
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), 'set GEMINI key'

W_PG = 0.5            # propose-ground RRF weight when fusing into the pool
K = 50               # LLM ranker head (== gate/serve)

# user_profile per (session,turn) aligned to cell-4 order (cell 4 doesn't keep it).
_dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
_ups = []
for sess in _dev:
    df = pd.DataFrame(sess['conversations']); up = sess.get('user_profile') or {}
    _ups += [up] * int((df['role'] == 'music').sum())
assert len(_ups) == len(queries)

# TURN-1 subset (Blind proxy) + wall mask
t1 = [i for i in range(len(queries)) if turn_numbers[i] == 1]
q  = [queries[i] for i in t1];  g = [golds[i] for i in t1]
cpool = [cs[i] for i in t1];    u = [user_ids[i] for i in t1];  cx = [ctx[i] for i in t1]
gc = [goal_categories[i] for i in t1]; gs = [goal_specificities[i] for i in t1]
prof = [_ups[i] for i in t1]
def _art(t):
    m = item_db.metadata_dict.get(t) or {}; a = m.get('artist_name')
    return a[0] if isinstance(a, list) and a else a
wall = np.array([_art(g[j]) not in {_art(t) for t in played[t1[j]]} for j in range(len(t1))])
print(f'[C] turn-1 n={len(t1)} wall={int(wall.sum())}')

# 1) propose-ground on the turn-1 subset (cold-firable: ignores history)
pg_ret = load_retrieval_module('propose_ground', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'pg_model': 'gemini-2.5-flash', 'n_proposals': 20, 'batch_size': 4,
                  'inner_dense': 'dense_metadata_qwen3_instruct'})
pg = pg_ret.batch_text_to_item_retrieval(q, topk=100, user_ids=u, batch_context=cx)

# 2) fuse union(cs, pg)
fused = RRF_MODEL.fuse_per_sub([cpool, pg], [1.0, W_PG], 60, 100)

def _recall(cands, k, mask=None):
    idx = range(len(g)) if mask is None else list(np.where(mask)[0])
    if not len(idx): return float('nan')
    return float(np.mean([1.0 if g[i] in cands[i][:k] else 0.0 for i in idx]))
print('--- recall (turn-1): all | WALL ---')
for nm, cd in [('cs (baseline)', cpool), ('union(cs,pg)', fused)]:
    print(f'  {nm:14s} @20 {_recall(cd,20):.4f} | {_recall(cd,20,wall):.4f}   '
          f'@100 {_recall(cd,100):.4f} | {_recall(cd,100,wall):.4f}')

# 3) LLM-rank cs vs union(cs,pg) -> turn-1 nDCG@20
rr = LLMListwiseReranker(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                         model_path='gemini-2.5-flash-lite', k=K)
def _llm_ndcg(cands, tag):
    rk = rr.rerank(q, cands, topk=20, user_ids=u, goal_categories=gc,
                   goal_specificities=gs, user_profiles_raw=prof)
    nd = ndcg_by_turn(rk, g, [1]*len(g), k=20)['overall']
    npar = rr.diagnostics['n_parsed']; hl = rr.diagnostics['head_len']
    vif = float(np.mean([(npar[j] or 0)/max(1, hl[j]) for j in range(len(q))]))
    print(f'  LLM-rank {tag:14s} turn-1 nDCG@20 = {nd:.4f}  (valid-index {vif:.2f})')
    return nd

print('--- turn-1 nDCG@20 ---')
print(f'  recall-only cs            = {ndcg_by_turn([c[:20] for c in cpool], g, [1]*len(g), k=20)["overall"]:.4f}')
nd_cs  = _llm_ndcg(cpool, 'cs')
nd_pg  = _llm_ndcg(fused, 'union(cs,pg)')
print(f'\nGATE: LLM(cs+pg)={nd_pg:.4f} vs LLM(cs)={nd_cs:.4f} (=0.2256 ref) -> '
      f'{"pg CONVERTS (+%.4f) -> fold pg in + retrain" % (nd_pg-nd_cs) if nd_pg-nd_cs>0.005 else "pg does NOT convert -> wall unreachable here"}')


In [ ]:
# 4-route) Segment-aware routing (no LLM, no OOM): cold queries (no played
# history) lean on CONTENT channels; warm queries lean on the SESSION channel.
# Opt-in use_segment_routing; the RRF layer picks per-query weights by history.
# Requires cell 4 (cs, queries, golds, turn_numbers, ctx, user_ids) + 4-strat.
from mcrs.eval_ndcg import recall_by_turn, format_by_turn
_ur = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS,
        CACHE_DIR, extra_config={'use_sasrec': True, 'w_sasrec': 1.0,
        'use_segment_routing': True})
_cr = _ur.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
print(format_by_turn(recall_by_turn(cs, golds, turn_numbers, 100), 'union+SASRec (baseline)'))
print(format_by_turn(recall_by_turn(_cr, golds, turn_numbers, 100), '+segment_routing'))


In [ ]:
# 4-route-ndcg) ANALYSIS: does segment routing convert to nDCG@20?
# Clean isolation — recall-only nDCG@20 of the FUSED top-20 order (NO reranker,
# so no train/serve confound from wrrf_rank). If routed > baseline here, routing
# improves the order nDCG sees -> worth a feature-rebuild + retrain to capture it
# downstream. Requires cell 4 (cs, golds, turn_numbers) + 4-route (_cr) + 4-strat.
print('=== recall-only nDCG@20: baseline vs segment routing (fused order) ===')
report_by_turn([c[:20] for c in cs],  'baseline fused (union+SASRec)')
report_by_turn([c[:20] for c in _cr], 'routed fused (segment routing)')


In [ ]:
# 4b) BUG #1 CONTRAST — decision-harness history vs SERVE history, same sessions.
#
# WHY every dev recall/nDCG number in this notebook is optimistic: cell 4 injects
# the REAL played list into history_tids (line 37 builds it from raw role=='music';
# line 41 puts it in ctx). But at SERVE, crs_baseline.batch_chat builds history_tids
# by filtering role=='music' (crs_baseline.py:603-606) on parser output that has
# ALREADY rewritten every music turn to role=='assistant'
# (run_inference_devset.py:40-44 / run_inference_blindset.py:31-41) -> always [].
# So SASRec's played-track sequence + the LGBM session-continuity features run
# EMPTY in production but FULL here. That is root cause B1.
#
# No GPU. Reuses `dev` from cell 4 (re-loads if missing). Parser/filter logic is
# copied VERBATIM from the serve code; id_to_metadata is stubbed (its TEXT output
# is irrelevant to the role/track_id filter).
import collections, pandas as pd
try:
    dev
except NameError:
    from datasets import load_dataset
    dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

def _meta(tid):
    return f'track metadata text for {tid}'   # prod expands music content to TEXT

def dev_parser_prior(prior_records):           # run_inference_devset.py:40-44
    ch = []
    for t in prior_records:
        if t['role'] == 'music':
            ch.append({'role': 'assistant', 'content': _meta(t['content'])})  # NO track_id
        else:
            ch.append({'role': t['role'], 'content': t['content']})
    return ch

def blind_parser_prior(prior_records):         # run_inference_blindset.py:31-41
    ch = []
    for t in prior_records:
        if t['role'] == 'music':
            ch.append({'role': 'assistant', 'content': _meta(t['content']), 'track_id': t['content']})
        else:
            ch.append({'role': t['role'], 'content': t['content']})
    return ch

def crs_history_tids(prior_history):           # crs_baseline.py:603-606 (VERBATIM)
    return [str(t.get('track_id') or t.get('content')) for t in prior_history
            if t.get('role') == 'music' and (t.get('track_id') or t.get('content'))]

st = collections.Counter()
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[df['turn_number'] < tn].to_dict('records')
        harness_played = [t['content'] for t in prior if t['role'] == 'music']  # cell 4, line 37
        if not harness_played:
            continue
        st['n'] += 1
        st['harness']     += 1   # full played list, by construction
        st['serve_dev']   += 1 if crs_history_tids(dev_parser_prior(prior))   else 0
        st['serve_blind'] += 1 if crs_history_tids(blind_parser_prior(prior)) else 0

n = st['n']
print(f'Dev prediction turns that HAVE played history (should use it): {n}\n')
print(f'history_tids NON-EMPTY, out of {n}:')
print(f'  decision harness  (nb74 cell 4, lines 37/41)  : {st["harness"]:5d}/{n}   <- full played list')
print(f'  SERVE, dev parser   (crs_baseline.batch_chat) : {st["serve_dev"]:5d}/{n}')
print(f'  SERVE, blind parser (crs_baseline.batch_chat) : {st["serve_blind"]:5d}/{n}')
print()
print('CONCLUSION: this notebook feeds the channels FULL session history; serve feeds')
print('them EMPTY. Every dev recall/nDCG number above was measured on a pipeline that')
print('production (run_inference_devset/blindset -> crs_baseline) does NOT run = root')
print('cause B1. Fix: crs_baseline.py:605 should match role in ("music","assistant");')
print('the dev parser must also attach track_id like the blind parser does.')
print()
print('(same_artist/session_cf have a separate track_id fallback in')
print(' played_tids_from_context -> see the standalone probe nb 76 for that nuance.)')


In [ ]:
# 4c) BUG #1 FIX VALIDATION — runs the REAL, now-fixed serve code path.
#
# Unlike cell 4b (which hardcodes the OLD logic to demonstrate the bug), this
# imports the fixed functions from the repo: run_inference_devset.chat_history_parser
# (now attaches track_id) + CRS_BASELINE._played_tids_for (now reuses
# played_tids_from_context). Run AFTER cell 1 has cloned the fixed branch.
# No GPU.
#
# EXPECTED: serve history_tids non-empty == N/N (was 0/N before the fix).
import pandas as pd, collections
from datasets import load_dataset
from run_inference_devset import chat_history_parser      # FIXED: attaches track_id
from mcrs.crs_baseline import CRS_BASELINE                # FIXED: _played_tids_for

try:
    dev
except NameError:
    dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

class _StubItemDB:
    def id_to_metadata(self, tid):
        return f'META:{tid}'   # content text is irrelevant to history_tids recovery
class _StubCRS:
    item_db = _StubItemDB()

# real played-id catalog + a bare CRS instance (no model load) to call the fix
catalog = set()
for item in dev:
    for t in item['conversations']:
        if t['role'] == 'music':
            catalog.add(str(t['content']))
crs = CRS_BASELINE.__new__(CRS_BASELINE)
crs._valid_catalog = catalog

st = collections.Counter()
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        if not len(df[(df['role'] == 'music') & (df['turn_number'] < tn)]):
            continue
        st['n'] += 1
        ch, _ = chat_history_parser(sess['conversations'], _StubCRS(), tn)  # FIXED parser
        if crs._played_tids_for(ch):                                       # FIXED method
            st['serve_fixed'] += 1

n = st['n']
print(f'Turns with played history: {n}')
print(f'SERVE history_tids NON-EMPTY (FIXED code path): {st["serve_fixed"]}/{n}')
print()
if st['serve_fixed'] == n:
    print('FIX CONFIRMED: serve now recovers the played history for every turn')
    print('(was 0/{} in cell 4b). SASRec sequence + LGBM session features + same_artist'.format(n))
    print('+ session_cf now receive the real played list in production.')
else:
    print('UNEXPECTED: some turns still empty — inspect before relying on the fix.')


In [ ]:
# 4d) MEASURE the bug #1 fix on dev nDCG@20 (offline, no submission needed).
#
# Runs the SAME union+SASRec -> lgbm_clean_full pipeline twice, isolating EXACTLY
# the history_tids fix: BROKEN = empty history (what serve did) vs FIXED = the
# real played list. Everything else held constant, so the delta is the fix's
# expected nDCG lift (SASRec sequence + LGBM session features).
#
# Requires cells 1, 3, 4 (queries, golds, played, user_ids, user_dialogs,
# turn_numbers, goal_categories/specificities, sas, RRF) + lgbm_clean_full
# (on Drive via cell 1, or trained in Stage 11). GPU. A few minutes on full dev.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')
weights = [s['weight'] for s in sas.subs]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

def score(history_per_query, tag):
    ctx = [{'history_tids': h, 'user_dialog': ud}
           for h, ud in zip(history_per_query, user_dialogs)]
    per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
    si = labels.index('sasrec_seq')
    fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
    efpc = []
    for qi, cands in enumerate(fused):
        rm = {tid: r + 1 for r, tid in enumerate(per_sub[si][qi])}
        efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
    esi = [{'played_tids': history_per_query[i], 'turn_number': turn_numbers[i],
            'prior_track_count': len(history_per_query[i])} for i in range(len(queries))]
    reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories, goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    sc = ndcg20(reranked)
    print(f'  {tag}: nDCG@20 = {round(sc, 4)}')
    return sc

print('=== Bug #1 fix impact on dev nDCG@20 (union+SASRec -> lgbm_clean_full) ===')
broken = score([[] for _ in queries], 'BROKEN serve (empty history) ')
fixed  = score(played,               'FIXED  serve (real played)   ')
print(f'  delta (fixed - broken)        : {round(fixed - broken, 4)}')
print()
print('This delta is the OFFLINE estimate of what the history_tids fix recovers on')
print('the serve pipeline. A positive delta => a real (not cosmetic) Blind-A gain is')
print('likely; ~0 => the fix is correctness-only on this pipeline. Blind-A is a')
print('different dataset/scorer, so treat this as directional, not exact.')


In [ ]:
# 4e) MEASURE the bug #2 fix on dev nDCG@20: played-track exclusion (offline).
#
# On the FIXED pipeline (real history), compares dev nDCG@20 WITHOUT vs WITH
# dropping already-played tracks from the top-20. The gold is ALWAYS a new track,
# so excluding played tracks can only promote real candidates -> provably
# non-decreasing for nDCG. Uses the same CRS_BASELINE._finalize_topk that ships.
# Requires cells 1, 3, 4 (+ lgbm_clean_full). GPU. A few minutes.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.crs_baseline import CRS_BASELINE

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')
weights = [s['weight'] for s in sas.subs]
_crs = CRS_BASELINE.__new__(CRS_BASELINE)            # bare instance for _finalize_topk
_crs._valid_catalog = set(item_db.metadata_dict.keys())

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

# FIXED pipeline: real history -> fused top-100 pool + reranked top-20
ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
si = labels.index('sasrec_seq')
fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
efpc = []
for qi, cands in enumerate(fused):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[si][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                     goal_categories=goal_categories, goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_features_per_candidate=efpc, extra_session_info=esi)

# OLD assembly (no played-exclusion) vs the shipped _finalize_topk (with it)
no_excl   = [_crs._finalize_topk(reranked[i], fused[i], set())            for i in range(len(queries))]
with_excl = [_crs._finalize_topk(reranked[i], fused[i], set(played[i]))   for i in range(len(queries))]

print('=== Bug #2 fix impact on dev nDCG@20 (played-track exclusion) ===')
a = ndcg20(no_excl);   print(f'  WITHOUT played-exclusion : nDCG@20 = {round(a, 4)}')
b = ndcg20(with_excl); print(f'  WITH    played-exclusion : nDCG@20 = {round(b, 4)}')
print(f'  delta (with - without)   : {round(b - a, 4)}')
print()
print('Provably non-decreasing (the gold is never a played track), so this delta')
print('is a free no-GPU nDCG gain that STACKS on top of the bug #1 fix.')


In [ ]:
# 4f) DIAGNOSTIC: does the reranker have UNUSED relevance signal? (no retrain)
#
# The workflow found the LGBM has no real query->track relevance feature — the
# rank_inv features all degenerate to 1/wrrf_rank (build_lgbm_features.py:370-375)
# and ce_score is a constant 0.0. Before building a dense-cosine feature, test
# whether the dense channel's own ordering of the ALREADY-RECALLED candidates
# separates golds better than the fused (wrrf) order. Rerank the fused top-100 by
# each channel's rank and score nDCG@20. Requires cells 1, 3, 4. Cheap, no retrain.
import math
from mcrs.retrieval_modules.rrf import RRF_MODEL

ctx = [{'history_tids': p, 'user_dialog': ud} for p, ud in zip(played, user_dialogs)]
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('=== Unused relevance signal for the reranker? dev nDCG@20 ===')
print(f'  wrrf fused order (baseline)        : {round(ndcg20([c[:20] for c in fused]), 4)}')
for ci, ch in enumerate(labels):
    out = []
    for qi, cands in enumerate(fused):
        rm = {tid: r for r, tid in enumerate(per_sub[ci][qi])}
        out.append(sorted(cands, key=lambda t: rm.get(t, 10**6)))  # missing -> bottom
    print(f'  rerank fused by {ch:22s}: {round(ndcg20(out), 4)}')
print()
print('Read: if a channel (especially the dense one) BEATS the wrrf baseline, its')
print('per-candidate ordering separates golds better than the fused rank -> a')
print('relevance feature from it would give the LGBM signal it currently lacks.')
print('NOTE: this tests RANK, not the cosine SCORE. A positive result is a strong')
print('GO; a flat/negative result is inconclusive (the score carries magnitude the')
print('rank throws away) -> then test the raw cosine before deciding.')


In [ ]:
# 4g) PHASE 0a — does the FINE-TUNED BGE-M3 add recall the union+SASRec misses?
# Additivity test, NO training. Reuses cell-4 globals. Run cells 1, 3, 4 first. GPU.
#
# CATALOG FORMAT NOTE (empirical, 2026-06-01): build_doc_text (5-field, pipe,
# original case) MEASURED BETTER than id_to_metadata (3-field) for this hub model
# -> standalone recall 0.4274 vs 0.3956. So the model was trained on a richer
# format; we use build_doc_text here. (The exact training format of this
# black-box hub model is unknown -> the clean fix is to RETRAIN bge_m3_ft with a
# format WE control. See project_bge_m3_ft_improvement_roadmap.)
# The .fmt marker invalidates the stale id_to_metadata pickle so it re-embeds once.
import os, pickle, numpy as np, torch, pandas as pd
from datasets import load_dataset
from mcrs.crs_baseline import build_retrieval_query
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.rrf import RRF_MODEL

BGE_HUB   = "OrRim123/recsys2026-bge-m3-music-v1-merged"
BGE_LABEL = "bge-m3-music-v1-merged"
TOPK = 100
FMT_TAG = "build_doc_text_v1"

safe_model = BGE_HUB.replace("/", "_")
emb_path = os.path.join(CACHE_DIR, "dense_local", safe_model, BGE_LABEL, "track_embeddings.pkl")
marker = emb_path + ".fmt"
fmt_ok = os.path.exists(marker) and open(marker).read().strip() == FMT_TAG
if (not os.path.exists(emb_path)) or (not fmt_ok):
    print("[0a] (re)embedding catalog in build_doc_text format (empirically best)...")
    from sentence_transformers import SentenceTransformer
    cat = load_dataset(ITEM_DB, split="all_tracks")
    FIELDS = ["track_name", "artist_name", "album_name", "tag_list", "release_date"]
    def _doc(row):
        parts = []
        for f in FIELDS:
            v = row.get(f)
            if v is None: continue
            if isinstance(v, list):
                if not v: continue
                v = ", ".join(str(x) for x in v)
            elif v == "": continue
            parts.append(f"{f}: {v}")
        return " | ".join(parts)
    tids = [r["track_id"] for r in cat]
    docs = [_doc(r) for r in cat]
    print("[0a] sample catalog doc:", docs[0][:160])
    dev_ = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    enc = SentenceTransformer(BGE_HUB, device=dev_)
    mat = enc.encode(docs, batch_size=64, normalize_embeddings=True,
                     convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    os.makedirs(os.path.dirname(emb_path), exist_ok=True)
    with open(emb_path, "wb") as f:
        pickle.dump({"track_ids": tids, "track_mat": mat}, f)
    with open(marker, "w") as mf:
        mf.write(FMT_TAG)
    del enc
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"[0a] wrote {emb_path}  (N={len(tids)}, dim={mat.shape[1]})")
else:
    print(f"[0a] reusing build_doc_text catalog embedding at {emb_path}")

dev = load_dataset("talkpl-ai/TalkPlayData-Challenge-Dataset", split="test")
struct_queries = []
for sess in dev:
    df = pd.DataFrame(sess["conversations"])
    goal = sess.get("conversation_goal") or {}
    goal_txt = (goal.get("listener_goal") or "").strip()
    up = sess.get("user_profile") or {}
    for _, music in df[df["role"] == "music"].iterrows():
        tn = int(music["turn_number"])
        prior = df[(df["turn_number"] < tn) |
                   ((df["turn_number"] == tn) & (df["role"] == "user"))]
        sm = []
        for _, t in prior.iterrows():
            role = t["role"]
            if role == "music":
                sm.append({"role": "assistant", "content": item_db.id_to_metadata(t["content"])})
            else:
                sm.append({"role": role, "content": t["content"]})
        struct_queries.append(build_retrieval_query(
            sm, mode="bge_m3_structured", goal_text=goal_txt,
            user_profile=up, max_history_turns=6))
assert len(struct_queries) == len(golds), (len(struct_queries), len(golds))
print(f"[0a] built {len(struct_queries)} bge_m3_structured queries")

bge = load_retrieval_module(
    "dense_metadata_bge_m3_ft_local", ITEM_DB, ["all_tracks"], CORPUS,
    CACHE_DIR, extra_config={"hub_repo": BGE_HUB})
bge_top = bge.batch_text_to_item_retrieval(struct_queries, topk=TOPK, user_ids=user_ids)

weights = [s["weight"] for s in sas.subs]
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
union_top = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, TOPK)

def _r(cands, k=TOPK):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, golds)]))
bge_recall   = _r(bge_top)
union_recall = _r(union_top)

def _aid(tid):
    a = item_db.metadata_dict.get(tid, {}).get("artist_id")
    return (a[0] if a else None) if isinstance(a, list) else a

union_missed = rescued = rescued_new_artist = hyp_hits = 0
for i, g in enumerate(golds):
    in_union = g in union_top[i]; in_bge = g in bge_top[i]
    if in_union or in_bge: hyp_hits += 1
    if not in_union:
        union_missed += 1
        if in_bge:
            rescued += 1
            paids = {_aid(t) for t in played[i]}; paids.discard(None)
            if _aid(g) not in paids: rescued_new_artist += 1
hyp_recall = hyp_hits / len(golds)
new_artist_pct = (rescued_new_artist / rescued * 100.0) if rescued else 0.0
rescue_share   = (rescued / union_missed * 100.0) if union_missed else 0.0

print("\n=== Phase 0a: BGE-M3-FT additivity (n=%d) — build_doc_text format ===" % len(golds))
print(f"  bge_m3_ft STANDALONE recall@{TOPK} : {bge_recall:.4f}")
print(f"  union+SASRec        recall@{TOPK}  : {union_recall:.4f}")
print(f"  hypothetical UNION  recall@{TOPK}  : {hyp_recall:.4f}  (+{hyp_recall - union_recall:.4f} over union)")
print(f"  rescued by bge_m3_ft               : {rescued}  ({rescue_share:.1f}% of union-missed)")
print(f"  of rescued, NEW-ARTIST             : {rescued_new_artist}  ({new_artist_pct:.1f}%)")
ADDITIVE = (rescue_share >= 5.0) and (new_artist_pct >= 50.0)
print("--- VERDICT:", "ADDITIVE (build into union; retrain LGBM with it)" if ADDITIVE else "REDUNDANT", "---")


In [ ]:
# 4h) Phase 0a-nDCG: does the bge_m3_ft recall gain CONVERT to nDCG@20?
# Fuses bge_m3_ft (from cell 4g) into the union, reranks with lgbm_clean_full,
# compares dev nDCG@20 to the union-only baseline. Requires cells 4 + 4g
# (uses per_sub, labels, weights, bge_top, recall_at). GPU.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

W_BGE = 0.6   # untuned; mirrors config 180/182's bge_m3_ft weight
per_sub_plus = per_sub + [bge_top]
weights_plus = weights + [W_BGE]
fused_base = RRF_MODEL.fuse_per_sub(per_sub,      weights,      sas.k, 100)
fused_plus = RRF_MODEL.fuse_per_sub(per_sub_plus, weights_plus, sas.k, 100)

si = labels.index('sasrec_seq')
def make_efpc(fused):
    out = []
    for qi, cands in enumerate(fused):
        rm = {tid: r + 1 for r, tid in enumerate(per_sub[si][qi])}
        out.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
    return out
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def rerank_score(fused, tag):
    rk = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                   goal_categories=goal_categories, goal_specificities=goal_specificities,
                   user_profiles_raw=[None] * len(queries),
                   extra_features_per_candidate=make_efpc(fused), extra_session_info=esi)
    sc = ndcg20(rk); print(f'  {tag}: nDCG@20 = {round(sc, 4)}'); return sc

print('=== Phase 0a-nDCG: does the bge_m3_ft recall gain convert? ===')
print(f'  recall@100  : union {round(recall_at(fused_base,100),4)} -> +bge {round(recall_at(fused_plus,100),4)}')
a = rerank_score(fused_base, 'union+SASRec        -> lgbm')
b = rerank_score(fused_plus, 'union+SASRec+bge_ft -> lgbm')
print(f'  delta (with bge - without)  : {round(b - a, 4)}')
print()
print('NOTE: lgbm_clean_full was trained WITHOUT a bge channel, so wrrf_rank shifts')
print('(mild train/serve skew) and W_BGE is untuned. A positive delta despite that is')
print('STRONG; flat/negative likely means the reranker must be retrained WITH the bge')
print('channel to lift the new (mostly new-artist) golds into the top-20.')


## Stage 2 — LGBM rerank: build features (with SASRec) + train (with vs without the SASRec feature)

In [ ]:
# 6) Build LGBM LambdaRank features from train sessions, WITH the SASRec channel
# (emits the real sasrec_rank_inv column). Two seeds -> train + val splits.
import os
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
os.makedirs(LGBM_DIR, exist_ok=True)
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
if os.path.exists(TRAIN_FEAT) and os.path.exists(VAL_FEAT):
    print('[lgbm] feature parquets present, skipping build')
else:
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TRAIN_FEAT}
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 400 --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VAL_FEAT}


In [ ]:
# 7) Train THREE LGBM models on the SAME feature build to isolate the in-sample
# model-derived feature leak (see project_sasrec_lgbm_feature_leak memory):
#   lgbm_sasrec_v1   : all columns (sasrec_rank_inv + cfbpr_score present)
#   lgbm_nosasrec_v1 : sasrec_rank_inv dropped (cfbpr_score still present)
#   lgbm_clean_v1    : BOTH sasrec_rank_inv AND cfbpr_score dropped  <-- LEAK TEST
# train_lgbm_ranker auto-selects every non-id column as a feature, so dropping a
# column is the clean one-axis control. Cheap (CPU, no-GPU) confirmatory test:
# if lgbm_clean_v1 matches/beats recall-only on dev (cell 9) while the others lose,
# the leak is confirmed as the reranker's whole problem.
import os, json
import pandas as pd
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRAIN_FEAT = f'{LGBM_DIR}/lgbm_train_sasrec.parquet'
VAL_FEAT   = f'{LGBM_DIR}/lgbm_val_sasrec.parquet'
TRAIN_NS   = f'{LGBM_DIR}/lgbm_train_nosasrec.parquet'
VAL_NS     = f'{LGBM_DIR}/lgbm_val_nosasrec.parquet'
TRAIN_CL   = f'{LGBM_DIR}/lgbm_train_clean.parquet'
VAL_CL     = f'{LGBM_DIR}/lgbm_val_clean.parquet'
# nosasrec control: drop only sasrec_rank_inv
for src, dst in [(TRAIN_FEAT, TRAIN_NS), (VAL_FEAT, VAL_NS)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in ['sasrec_rank_inv'] if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built no-sasrec control parquets')
# clean (leak test): drop BOTH model-derived leaked features
LEAKED = ['sasrec_rank_inv', 'cfbpr_score']
for src, dst in [(TRAIN_FEAT, TRAIN_CL), (VAL_FEAT, VAL_CL)]:
    df = pd.read_parquet(src)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(dst, index=False)
print('[lgbm] built clean (no-leak) parquets \u2014 dropped', LEAKED)

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_FEAT} --val-features {VAL_FEAT} \
    --output-dir {LGBM_DIR}/lgbm_sasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_NS} --val-features {VAL_NS} \
    --output-dir {LGBM_DIR}/lgbm_nosasrec_v1
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} --val-features {VAL_CL} \
    --output-dir {LGBM_DIR}/lgbm_clean_v1

print('\n=== Stage 2 held-out val nDCG@20 (LGBM internal \u2014 leak-inflated, see cell 9 for honest dev) ===')
for name in ['lgbm_clean_v1', 'lgbm_nosasrec_v1', 'lgbm_sasrec_v1']:
    meta = json.load(open(f'{LGBM_DIR}/{name}/metadata.json'))
    print('  ' + name + ': val_ndcg@20=' + str(round(meta['best_val_ndcg20'], 4)) +
        '  (' + str(len(meta['features'])) + ' features)')


## Stage 3 — end-to-end DEV nDCG@20: recall -> rerank

In [ ]:
# 9) End-to-end on dev: union+SASRec recall@100 -> LGBM rerank top-20 -> nDCG@20.
# All LGBM models rerank the SAME union+SASRec pool, so this isolates the rerank
# feature set. The SASRec per-candidate rank is fed exactly as crs_baseline does in
# prod; LGBM_RERANKER only uses features listed in its own metadata, so the clean
# model harmlessly ignores efpc's sasrec_rank and skips cfbpr_score.
# LEAK TEST READING: recall-only is the bar (0.1473 in the run that found the leak).
#   - lgbm_clean_v1 (no leaked feats) >= recall-only  => leak WAS the problem; this
#     leak-free reranker is shippable with NO GPU. OOF only needed to ADD sasrec
#     value on top.
#   - lgbm_clean_v1 still < recall-only               => leak isn't the whole story;
#     OOF would not have helped alone. Investigate label sparsity / wrrf_rank
#     contamination / train-set size next.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2)
                break
    return s / len(golds)

print('=== Stage 3 end-to-end DEV (n=' + str(len(golds)) + ') ===')
print('  recall@100 pool ceiling      :', round(recall_at(fused100, 100), 4))
recall_only = ndcg20([r[:20] for r in fused100])
print('  nDCG@20 recall-only (no rerank):', round(recall_only, 4), '  <-- bar to beat')
for name, sub in [('LGBM clean (no leaked feats)', 'lgbm_clean_v1'),
                  ('LGBM (no sasrec feat)      ', 'lgbm_nosasrec_v1'),
                  ('LGBM (+ sasrec feat)       ', 'lgbm_sasrec_v1')]:
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                       model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/{sub}')
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    score = ndcg20(reranked)
    flag = '  BEATS recall-only' if score >= recall_only else ''
    print('  nDCG@20 ' + name + ' :', round(score, 4), flag)


## Stage 4 — Blind-A submission (separate logic)

Runs the full pipeline via config 194 (union+SASRec -> lgbm_clean_full (leak-free) -> v5-kto responder) over the 80 Blind-A queries and packages prediction.json for CodaBench. This is the LONG cell (~35-65 min). Blind-A nDCG is server-scored — there are no local golds; use Stage 3 as the local signal before submitting.

Requires Stage 11 to have written lgbm_clean_full to the Drive cache (config 194 points at it).

In [ ]:
# 11) Blind-A inference -> prediction.json -> zip for CodaBench.
# Ships config 194 = union+SASRec -> lgbm_clean_full (leak-free, 15k-session, the
# nb74 Stage 11 best, dev nDCG@20 0.1623) -> v5-kto responder. NOT config 191
# (leaked reranker lgbm_sasrec_v1, dev 0.1194 — do not ship).
import os
TID = '194-union-sasrec-lgbm-cleanfull-v5kto-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
# config 194 reranker_model_path must resolve to lgbm_clean_full (built in Stage 11).
assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full/booster.txt'), \
    'Run Stage 11 first — lgbm_clean_full booster missing.'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

import json
preds = json.load(open(PRED_PATH))
n = len(preds) if isinstance(preds, list) else len(preds)
print('[blindA] prediction entries:', n)
assert n == 80, f'expected 80 Blind-A entries, got {n} — DO NOT submit'
print('[blindA] sample keys:', list(preds[0].keys()))

# Optional strict precheck (catalog membership + schema):
#   !python scripts/precheck_prediction.py {PRED_PATH}

import zipfile, datetime
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-194.zip'  # short name: CodaBench caps the submission name at 64 chars
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA] submission zip ready:', zip_path)
print('[blindA] upload to https://www.codabench.org/competitions/ and append scores via scripts/blind_a_score_tracker.py')


# Hand off to nb80: copy the FRESH prediction (carries the SASRec user_dialog
# serve fix in predicted_track_ids) to the stable Drive path nb80 reads as SRC,
# overwriting any older candidates there.
import shutil
DRIVE_PRED = '/content/drive/MyDrive/recsys2026/194-union-sasrec-lgbm-cleanfull-v5kto-blindA.json'
os.makedirs(os.path.dirname(DRIVE_PRED), exist_ok=True)
shutil.copy(PRED_PATH, DRIVE_PRED)
print('[blindA] candidates copied to Drive for nb80 (responder swap):', DRIVE_PRED)


In [ ]:
# 11-relev) Blind-A inference for config 197 = lgbm_relev (Tier-2 #4.1 relevance
# features + #4.3 bagging + holdout early-stop). Dev nDCG@20 turn-1 0.1819 (beats
# config 194's 0.1644 and the 2k 0.1712). The relevance features auto-inject at
# serve (crs_baseline builds a RelevanceScorer when the model lists them). Requires
# 12a (lgbm_relev trained). SUBMIT ONLY if you accept it as the new best.
import os
TID = '197-union-sasrec-lgbm-relev-v5kto-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_relev/metadata.json'), \
    'Run 12a first — lgbm_relev missing. (It is the sibling of lgbm_clean_full.)'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

import json
preds = json.load(open(PRED_PATH))
n = len(preds) if isinstance(preds, list) else len(preds)
print('[blindA] prediction entries:', n)
assert n == 80, f'expected 80 Blind-A entries, got {n} — DO NOT submit'
print('[blindA] sample keys:', list(preds[0].keys()))

import zipfile, datetime
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-197.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA] submission zip ready:', zip_path)


In [ ]:
# 11b) Blind-A inference for config 195 (top_n_for_prompt = 3).
# IDENTICAL pipeline to config 194 (union+SASRec -> lgbm_clean_full -> v5-kto)
# EXCEPT the responder sees the top-3 retrieved tracks and explains the one it
# can describe best, instead of being forced to explain the single nDCG-optimal
# (often new-artist) track at rank 1. Goal: recover the LLM axis (194 with the
# bug fixes scored LLM 2.35 vs 2.70) WITHOUT giving back the nDCG gain (0.31),
# since the scored top-20 list is unchanged. Run cells 1, 3 first.
import os
TID = '195-union-sasrec-lgbm-cleanfull-v5kto-topn3-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'
assert os.path.exists(f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full/booster.txt'), \
    'Run Stage 11 first — lgbm_clean_full booster missing.'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

import json
preds = json.load(open(PRED_PATH))
n = len(preds) if isinstance(preds, list) else len(preds)
print('[blindA-195] prediction entries:', n)
assert n == 80, f'expected 80 Blind-A entries, got {n} — DO NOT submit'
print('[blindA-195] sample keys:', list(preds[0].keys()))

import zipfile, datetime
zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-195-topn3.zip'  # short name: CodaBench caps the submission name at 64 chars
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA-195] submission zip ready:', zip_path)
print('[blindA-195] upload to https://www.codabench.org/competitions/ and append scores via scripts/blind_a_score_tracker.py')


In [ ]:
# 11d) Blind-A inference for config 198 = union+SASRec -> LLM LISTWISE (Gemini,
# Track A) -> v5-kto responder. Track A converted on the dev gate (# 12d-llm-listwise):
# turn-1 nDCG@20 0.1763 (recall-only) -> 0.2256 (+0.0493), valid-index 0.823. This
# ships it to the Blind-A leaderboard. Needs GEMINI_API_KEY in os.environ (load from
# Colab Secrets via userdata.get) + a `git pull` for the llm_listwise dispatch +
# config 198 + the load_crs_baseline fix. ~$0.05 of Gemini for the 80 queries.
import os, json, zipfile, datetime
assert os.environ.get('GEMINI_API_KEY') or os.environ.get('GOOGLE_API_KEY'), \
    'GEMINI key not in os.environ — load it from Colab Secrets: ' \
    "os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')"
TID = '198-union-sasrec-llm-listwise-v5kto-blindA'
PRED_PATH = f'/content/recsys2026/music-crs-baselines/exp/inference/blindset_A/{TID}.json'

%cd /content/recsys2026/music-crs-baselines
!python run_inference_blindset.py --tid {TID} --batch_size 8 2>&1 | tail -60
%cd /content/recsys2026

preds = json.load(open(PRED_PATH))
print('[blindA-198] entries:', len(preds), '| sample keys:', list(preds[0].keys()))
assert len(preds) == 80, f'expected 80 Blind-A entries, got {len(preds)} - DO NOT submit'

zip_path = f'/content/drive/MyDrive/recsys2026_submissions/{datetime.date.today().isoformat()}-blindA-198-llm.zip'
os.makedirs(os.path.dirname(zip_path), exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_PATH, arcname='prediction.json')   # MUST be at zip root for CodaBench
print('[blindA-198] submission zip ready:', zip_path)
print('[blindA-198] upload to https://www.codabench.org/competitions/ ; append via scripts/blind_a_score_tracker.py')


## Stage 5 - OOF leak-free test: does cross-fit sasrec_rank_inv beat the clean reranker?

The clean reranker (0.1558) drops sasrec_rank_inv entirely. This stage rebuilds
that feature LEAK-FREE via K-fold out-of-fold cross-fitting (each fold scored by
a SASRec that never trained on it), retrains the LGBM, and compares on dev:
- if lgbm_oof_v1 > 0.1558  -> leak-free SASRec rank ADDS value; keep it
- if lgbm_oof_v1 <= 0.1558 -> SASRec rank is dead even when honest; drop it
COST: K SASRec retrains (~25-40 min each on Blackwell) + 2K feature builds.

In [ ]:
# OOF (out-of-fold) cross-fit of sasrec_rank_inv, then retrain LGBM + compare.
# Requires cell 1 (CACHE_DIR/ITEM_DB/CORPUS) and cell 4 (sas, queries, user_ids,
# ctx, played, golds, turn_numbers, goal_categories, goal_specificities).
# Idempotent: each heavy step skips if its artifact exists, so re-running after
# an interruption does not redo finished folds.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

K = 5  # OOF folds = K SASRec retrains. Lower K is cheaper but weaker per-fold model.
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
SAS_DIR  = f'{CACHE_DIR}/retrieval_v2/sasrec'

# 1) Train K SASRec models, each HOLDING OUT fold k from training.
for k in range(K):
    out = f'sasrec_oof_{K}_{k}'
    if os.path.exists(f'{SAS_DIR}/{out}/sasrec.pt'):
        print(f'[oof] SASRec {out} present, skip'); continue
    !cd /content/recsys2026 && python -u scripts/train_sasrec.py \
        --cache-dir {CACHE_DIR} --out {out} \
        --d 256 --max-seq 50 --epochs 5 --batch-size 256 --lr 1e-3 \
        --oof-fold {k} --oof-num-folds {K}

# 2) Build leak-free feature parquets fold by fold (fold k scored by
#    sasrec_oof_K_k, which never saw fold k). Same seed/n-sessions as cell 6.
train_parts, val_parts = [], []
for k in range(K):
    tp = f'{LGBM_DIR}/oof_train_{K}_{k}.parquet'
    vp = f'{LGBM_DIR}/oof_val_{K}_{k}.parquet'
    if not os.path.exists(tp):
        !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
            --n-sessions 2000 --topk 100 --seed 42 \
            --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_oof_{K}_{k} \
            --oof-fold {k} --oof-num-folds {K} \
            --cache-dir {CACHE_DIR} --out {tp}
    if not os.path.exists(vp):
        !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
            --n-sessions 400 --topk 100 --seed 7 \
            --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_oof_{K}_{k} \
            --oof-fold {k} --oof-num-folds {K} \
            --cache-dir {CACHE_DIR} --out {vp}
    train_parts.append(tp); val_parts.append(vp)

# 3) Concat per-fold parquets, drop cfbpr_score (permanently leaked, not OOF-able).
TRAIN_OOF = f'{LGBM_DIR}/lgbm_train_oof.parquet'
VAL_OOF   = f'{LGBM_DIR}/lgbm_val_oof.parquet'
tr = pd.concat([pd.read_parquet(x) for x in train_parts], ignore_index=True)
va = pd.concat([pd.read_parquet(x) for x in val_parts], ignore_index=True)
assert 'sasrec_rank_inv' in tr.columns, 'OOF train parquet missing sasrec_rank_inv'
tr.drop(columns=[c for c in ['cfbpr_score'] if c in tr.columns]).to_parquet(TRAIN_OOF, index=False)
va.drop(columns=[c for c in ['cfbpr_score'] if c in va.columns]).to_parquet(VAL_OOF, index=False)
print(f'[oof] concatenated {K} folds -> train {len(tr)} rows, val {len(va)} rows')

# 4) Retrain LGBM on the OOF features (sasrec_rank_inv now honest).
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_OOF} --val-features {VAL_OOF} \
    --output-dir {LGBM_DIR}/lgbm_oof_v1

# 5) Compare on dev. dev is the test split -> SASRec never trained on it, so the
#    dev sasrec rank fed via efpc is already honest for every model.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 5 OOF comparison DEV (n=' + str(len(golds)) + ') ===')
recall_only = ndcg20([r[:20] for r in fused100])
print('  recall-only (no rerank)      :', round(recall_only, 4))
scores = {}
for name, sub in [('LGBM clean_v1  (2k, control) ', 'lgbm_clean_v1'),
                  ('LGBM clean_full(15k, shipped)', 'lgbm_clean_full'),
                  ('LGBM OOF       (2k, +sasrec) ', 'lgbm_oof_v1')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing, skipped)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    sc = ndcg20(reranked)
    scores[sub] = ndcg20(reranked)
    print('  ' + name + ' :', round(scores[sub], 4))
oof = scores.get('lgbm_oof_v1'); cv1 = scores.get('lgbm_clean_v1'); cf = scores.get('lgbm_clean_full')
print('\n-- DECISION (OOF at 2k) --')
if oof is not None and cv1 is not None:
    d = oof - cv1
    print('  feature test : OOF-2k %+.4f vs clean-2k (%s)' % (d, 'HELPS' if d > 0.002 else 'inert/redundant'))
if oof is not None and cf is not None:
    print('  ship test    : OOF-2k %+.4f vs clean-full 0.1623 shipped' % (oof - cf))
    print('  -> rebuild OOF at 15k ONLY if 2k feature test HELPS and reaches clean-full;')
    print('     else honest sasrec rank is redundant with wrrf_rank -> drop, keep clean_full.')


## Stage 6 - recall ceiling diagnostic: WHERE do we lose nDCG?

nDCG@20 is capped by recall@100 (0.4961). The reranker only reorders what recall
surfaced, so to climb the leaderboard we must raise recall, not rerank harder.
This cell (no GPU, reuses cell-4 data) breaks the ceiling down:
- per-channel miss rate (which channel finds what)
- ALL-channel miss = the hard wall no reranker can cross
- new-artist share of misses (cold-start / content gap)
- in-pool-but-past-100 (cheap recall@larger-K headroom)
The numbers pick the next lever: content modality vs wider pool vs channel swap.

In [ ]:
# Recall ceiling diagnostic. Requires cell 4 (sas, queries, user_ids, ctx,
# golds, played, item_db, recall_at, cs).
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
N = len(golds)
print(f'=== Stage 6 recall ceiling diagnostic (dev n={N}) ===')
print(f'channels: {labels}')

# 1) per-channel recall@100 (did this channel surface the gold at all?)
print('\n-- per-channel recall@100 (gold present in that channel top-100) --')
in_chan = [[False] * N for _ in labels]
for s in range(len(labels)):
    for q in range(N):
        in_chan[s][q] = golds[q] in set(per_sub[s][q])
    print(f'  {labels[s]:24s}: {sum(in_chan[s]) / N:.4f}')

# 2) union ceiling (any channel has it) — should track recall@100 of the fusion
any_chan = [any(in_chan[s][q] for s in range(len(labels))) for q in range(N)]
union_ceiling = sum(any_chan) / N
print(f'\n  UNION ceiling (any channel)      : {union_ceiling:.4f}')
print(f'  fused recall@100 (cs, post-RRF)  : {recall_at(cs, 100):.4f}')

# 3) the hard wall: golds NO channel surfaced
miss_idx = [q for q in range(N) if not any_chan[q]]
print(f'\n  ALL-CHANNEL MISS (hard wall)     : {len(miss_idx)}/{N} = {len(miss_idx)/N:.4f}')
print('  ^ no reranker can ever recover these; only a NEW recall signal can.')

# 4) new-artist analysis: of the all-channel misses, how many have an artist
#    that never appeared in the session history (cold-start / content gap)?
def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {})
    a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

new_artist_miss = 0
known_artist_miss = 0
for q in miss_idx:
    g_art = artist_of(golds[q])
    hist_arts = {artist_of(t) for t in played[q]}
    if g_art is not None and g_art in hist_arts:
        known_artist_miss += 1
    else:
        new_artist_miss += 1
if miss_idx:
    print(f'\n  of the {len(miss_idx)} hard-wall misses:')
    print(f'    new-artist (not in session history): {new_artist_miss} '
          f'({new_artist_miss/len(miss_idx):.1%})')
    print(f'    known-artist (in history, still missed): {known_artist_miss} '
          f'({known_artist_miss/len(miss_idx):.1%})')

# 5) cheap headroom: golds in the fused pool but ranked past 100 would need a
#    wider pool. Compare recall at 100 vs 200/500 of the fusion.
weights = [s['weight'] for s in sas.subs]
for wider in (200, 500):
    fused_wide = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, wider)
    print(f'  fused recall@{wider:<4d}              : {recall_at(fused_wide, wider):.4f}')
print('  ^ gain from 100 to 200/500 = cheap recall headroom (just widen topk).')

print('\n-- READING --')
print('  high new-artist share  -> add a content channel (lyrics modality) / stronger content')
print('  big jump at @200/@500  -> widen the candidate pool (nearly free)')
print('  one channel dominates  -> rebalance or replace a dead channel')


## Stage 7 - fix the content channel: dense_metadata_qwen3 A/B (no GPU train)

Stage 6 found dense_metadata_qwen3 at recall@100=0.0894 (near-dead) while the
new-artist hard wall is 98.8% of misses -- the content channel is exactly what
should catch those, and it's broken. Two suspected causes:
  P1 instruct prefix: the union uses dense_metadata_qwen3 (instruct=None), but
     the encoder Qwen3-Embedding-0.6B is ASYMMETRIC and needs a query instruct
     prefix. dense_metadata_qwen3_instruct applies it.
  P2 query pollution: cell 4's query is id_to_metadata text -> 'track_id: <uuid>,
     ...' for every prior music turn. UUIDs are noise to an embedder (bm25
     tolerates it; dense averages it in). user_dialog (user-turns only) is clean.
This A/B isolates each on dev recall@100. Track embeddings are precomputed, so
only the QUERY side re-encodes (minutes, no training). The winner is a one-line
union spec change (swap to _instruct and/or feed the clean query).

In [ ]:
# Dense channel A/B. Requires cell 4 (queries, user_ids, golds, recall_at,
# user_dialogs, ITEM_DB, CORPUS, CACHE_DIR).
from mcrs.retrieval_modules import load_retrieval_module

# Clean query = user-turns-only dialog (no uuid/metadata pollution). cell 4 built
# user_dialogs alongside queries; fall back to queries if absent.
clean_q = user_dialogs if 'user_dialogs' in dir() else queries

def dense_recall(retr_type, q_list, label):
    d = load_retrieval_module(retr_type, ITEM_DB, ['all_tracks'], CORPUS,
                              CACHE_DIR, extra_config={})
    cand = d.batch_text_to_item_retrieval(q_list, topk=100, user_ids=user_ids)
    r20 = recall_at(cand, 20)
    r100 = recall_at(cand, 100)
    print(f'  {label:38s}: @20={r20:.4f} @100={r100:.4f}')
    return r100

print('=== Stage 7 dense channel A/B (dev recall@100) ===')
print('  baseline reference: dense in union today = 0.0894 @100\n')
a = dense_recall('dense_metadata_qwen3',          queries, 'A raw + polluted query (reproduce)')
b = dense_recall('dense_metadata_qwen3_instruct', queries, 'B + instruct prefix (P1)')
c = dense_recall('dense_metadata_qwen3_instruct', clean_q, 'C instruct + clean user-dialog (P1+P2)')

best = max([('A', a), ('B', b), ('C', c)], key=lambda kv: kv[1])
print(f'\n  winner: {best[0]} @100={best[1]:.4f}  (vs current 0.0894)')
print('  A->B gain = instruct-prefix effect; B->C gain = clean-query effect.')
print('  Apply the winner to the union: _wrrf_union_v1_specs dense sub ->')
print('  dense_metadata_qwen3_instruct' + (' + feed clean query' if best[0] == 'C' else ''))


## Stage 8 - does the dense fix lift the UNION? (the decision gate)

Stage 7 doubled the dense channel in isolation (0.0894 -> 0.1789 instruct). But
the union recall@100 (0.4961) only rises if dense's now-working hits include
golds the OTHER channels miss -- especially new-artist golds (98.8% of the hard
wall). This cell measures that directly:
  - union recall@100 with dense FIXED (instruct) vs the 0.4961 baseline
  - of the OLD hard-wall misses, how many the fixed dense now rescues
  - how many of those rescues are new-artist (the wall we care about)
This is the gate: a real union lift + new-artist rescues = dense is the lever and
a STRONGER embedder (Qwen3-Embedding-4B on the A100-40GB) will compound. A flat
union = dense hits are redundant with bm25/sasrec; don't spend GPU on 4B.

In [ ]:
# Stage 8: union dense-fix recall gate. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at) and the dense-instruct fix in the factory.
from mcrs.retrieval_modules import load_retrieval_module

# Rebuild the union TWO ways: dense_instruct on (new default) vs off (old raw).
sas_fixed = load_retrieval_module(
    'wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0})  # dense_instruct defaults True
sas_raw = load_retrieval_module(
    'wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
    extra_config={'use_sasrec': True, 'w_sasrec': 1.0, 'dense_instruct': False})

cs_fixed = sas_fixed.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
cs_raw   = sas_raw.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)

N = len(golds)
r_raw  = recall_at(cs_raw, 100)
r_fix  = recall_at(cs_fixed, 100)
print('=== Stage 8 union recall@100: dense fix gate (dev n=%d) ===' % N)
print('  union (dense RAW, old)     : %.4f' % r_raw)
print('  union (dense INSTRUCT, new): %.4f' % r_fix)
print('  delta                      : %+.4f' % (r_fix - r_raw))

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {})
    a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

# Of golds the OLD union missed @100, how many does the FIXED union now rescue,
# and how many of those are new-artist (not in session history)?
old_miss = [q for q in range(N) if golds[q] not in set(cs_raw[q][:100])]
rescued = [q for q in old_miss if golds[q] in set(cs_fixed[q][:100])]
rescued_new_artist = 0
for q in rescued:
    g = artist_of(golds[q]); hist = {artist_of(t) for t in played[q]}
    if not (g is not None and g in hist):
        rescued_new_artist += 1
print('\n  old-union misses @100        : %d' % len(old_miss))
print('  rescued by dense fix         : %d' % len(rescued))
if rescued:
    print('    of which new-artist        : %d (%.1f%%)' %
          (rescued_new_artist, 100.0 * rescued_new_artist / len(rescued)))

print('\n-- DECISION --')
if r_fix - r_raw >= 0.005:
    print('  union LIFTED by the free fix -> dense is a live lever.')
    print('  -> a stronger embedder (Qwen3-Embedding-4B, A100-40GB) should compound.')
else:
    print('  union ~flat -> dense hits are largely redundant with bm25/sasrec.')
    print('  -> 4B upgrade likely low ROI; look elsewhere for new-artist recall.')


## Stage 9 - wider pool: convert latent recall into nDCG (GAP 1)

Stage 6 showed fused recall@100=0.4961 but @500=0.5675 -- +0.0714 of golds are
ALREADY surfaced at rank 100-500, just truncated by the top-100 pool. The reranker
(lgbm_clean_v1) reorders whatever pool it's given, so feeding it 200/500 candidates
lets it pull those deeper golds into the top-20 -- IF its features rank them well.
This cell reranks the SAME union+SASRec rankings at pool sizes {100,200,500} and
reports dev nDCG@20. Free on retrieval (re-fuses cached per-sub rankings).
  rising nDCG with pool size -> latent recall converts; widen the prod pool.
  flat/falling -> reranker can't surface deep golds; pool width won't help nDCG.

In [ ]:
# Stage 9: wider-pool nDCG. Requires cell 4 (sas, queries, user_ids, ctx, golds,
# played, turn_numbers, goal_categories, goal_specificities) + lgbm_clean_v1.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

# Per-sub rankings once (cached); re-fuse at each pool size. sas already has the
# instruct dense fix (cell 4 default).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
sidx = labels.index('sasrec_seq')

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

def recall_at_pool(fused, k):
    return sum(1.0 for f, g in zip(fused, golds) if g in set(f[:k])) / len(golds)

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_v1')
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

print('=== Stage 9 wider-pool nDCG@20 (dev n=%d, lgbm_clean_v1) ===' % len(golds))
print('  recall-only top-20 of fused@100 baseline = ndcg ref\n')
print('  %-6s %-12s %-12s' % ('pool', 'recall@pool', 'nDCG@20'))
for POOL in (100, 200, 500):
    fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, POOL)
    efpc = []
    for qi, cands in enumerate(fused):
        rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
        efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
    reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  %-6d %-12.4f %-12.4f' % (POOL, recall_at_pool(fused, POOL), ndcg20(reranked)))

print('\n-- READING --')
print('  nDCG rises with pool -> latent recall converts; set prod retrieval_topk higher.')
print('  nDCG flat/falls      -> reranker cannot surface deep golds; pool width is not the lever.')


## Stage 10 - reranker discrimination diagnostic: WHY can't it rank golds?

Stage 9 flipped the diagnosis: recall rose +0.0726 (pool 100->500) but nDCG@20
stayed flat 0.1558. The reranker can't surface deep golds. This cell measures WHY
(no GPU): lgbm_clean_v1 feature gains, and for golds PRESENT in pool@100 their rank
AFTER rerank + raw->reranked movement + net top-20 capture. If present-golds land
at high median rank, no feature separates gold from noise -> need data/features.

In [ ]:
# Stage 10: reranker discrimination diagnostic. Requires cell 4 (sas, queries,
# user_ids, ctx, golds, played, turn_numbers, goal_categories, goal_specificities).
import json, numpy as np
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
meta = json.load(open(f'{LGBM_DIR}/lgbm_clean_v1/metadata.json'))
print('=== Stage 10 reranker diagnostic (lgbm_clean_v1) ===')
print('  features:', len(meta['features']), '| best_iter:', meta.get('best_iteration'),
      '| internal val nDCG@20:', round(meta.get('best_val_ndcg20', 0), 4))

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
sidx = labels.index('sasrec_seq')
POOL = 100
fused = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, POOL)
efpc = []
for qi, cands in enumerate(fused):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{LGBM_DIR}/lgbm_clean_v1')
reranked = rr.rerank(queries, fused, topk=POOL, user_ids=user_ids,
                     goal_categories=goal_categories,
                     goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_features_per_candidate=efpc, extra_session_info=esi)

raw_ranks, new_ranks = [], []
for qi in range(len(golds)):
    g = golds[qi]
    if g in fused[qi]:
        raw_ranks.append(fused[qi].index(g) + 1)
        new_ranks.append(reranked[qi].index(g) + 1 if g in reranked[qi] else POOL + 1)
raw_ranks = np.array(raw_ranks); new_ranks = np.array(new_ranks)
nn = len(raw_ranks)
print(f'\n  golds present in pool@{POOL}: {nn} ({nn/len(golds):.1%} of all)')
print(f'  raw fused rank : median={int(np.median(raw_ranks))} mean={raw_ranks.mean():.1f} top20={(raw_ranks<=20).mean():.3f}')
print(f'  reranked rank  : median={int(np.median(new_ranks))} mean={new_ranks.mean():.1f} top20={(new_ranks<=20).mean():.3f}')
print(f'  present-gold movement: up={(new_ranks<raw_ranks).mean():.3f} down={(new_ranks>raw_ranks).mean():.3f} same={(new_ranks==raw_ranks).mean():.3f}')

print('\n  -- feature gains (lgbm_clean_v1) --')
gains = rr.booster.feature_importance(importance_type='gain')
for name, g in sorted(zip(rr.features, gains), key=lambda kv: -kv[1])[:15]:
    print(f'    {name:24s}: {g:.1f}')

print('\n-- READING --')
print('  reranked top20 >> raw top20  -> reranker helps; cap is elsewhere.')
print('  reranked top20 ~= raw top20  -> reranker near-inert; preserves fused order.')
print('  present-golds at high median -> no feature separates gold from noise')
print('     => more data / a stronger relevance feature is the lever.')


## Stage 11 - full-scale clean LGBM: does more training data help discrimination?

Stage 9 showed the reranker can't rank surfaced golds; lgbm_clean_v1 was trained on
only 2000 sessions with single-positive labels. This stage rebuilds the clean
feature set at FULL scale (15000 train / 2000 val sessions) and retrains, to test
whether the weak discrimination is a data-starvation problem.
  - feature build keeps the SASRec channel in the union (so the training candidate
    pool matches the serve pool) but DROPS the leaked columns (cfbpr_score,
    sasrec_rank_inv) -> identical clean feature set to lgbm_clean_v1.
  - compare dev nDCG@20: lgbm_clean_v1 (2k, 0.1558) vs lgbm_clean_full (15k).
COST: the feature build runs the union per train turn (~120k turns; dense re-encodes
new queries on GPU). Idempotent: skips if the full parquets already exist.

In [ ]:
# Stage 11: full-scale clean LGBM. Requires cell 1/3 (CACHE_DIR/ITEM_DB/CORPUS)
# and cell 4 (sas, queries, user_ids, ctx, golds, played, turn_numbers,
# goal_categories, goal_specificities).
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
N_TRAIN, N_VAL = 15000, 2000
TRAIN_FULL = f'{LGBM_DIR}/lgbm_train_full_sasrec.parquet'
VAL_FULL   = f'{LGBM_DIR}/lgbm_val_full_sasrec.parquet'

# 0) FORCE-REBUILD so the SASRec dialog-parity fix (Tier-0 #2) takes effect:
#    the build steps below skip when the parquet exists, which would reuse the
#    OLD skewed features. Delete the cached parquets first.
for _f in [TRAIN_FULL, VAL_FULL,
           f'{LGBM_DIR}/lgbm_train_full_clean.parquet',
           f'{LGBM_DIR}/lgbm_val_full_clean.parquet']:
    if os.path.exists(_f):
        os.remove(_f); print('[stage11] removed stale', _f)

# 1) Build full-scale features WITH the sasrec channel (matches serve pool).
if not os.path.exists(TRAIN_FULL):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions {N_TRAIN} --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TRAIN_FULL}
if not os.path.exists(VAL_FULL):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions {N_VAL} --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VAL_FULL}

# 2) Drop the leaked model-derived columns -> clean full feature set.
TRAIN_CL = f'{LGBM_DIR}/lgbm_train_full_clean.parquet'
VAL_CL   = f'{LGBM_DIR}/lgbm_val_full_clean.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TRAIN_FULL, TRAIN_CL), (VAL_FULL, VAL_CL)]:
    df = pd.read_parquet(s)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[lgbm] built full clean parquets (dropped', LEAKED, ')')

# 3) Train lgbm_clean_full.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} --val-features {VAL_CL} \
    --output-dir {LGBM_DIR}/lgbm_clean_full

# 4) Compare on dev nDCG@20 vs lgbm_clean_v1 (2k).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 11 full-scale clean LGBM, DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only baseline:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('lgbm_clean_v1 (2k sessions)', 'lgbm_clean_v1'),
                  ('lgbm_clean_full (%dk sessions)' % (N_TRAIN // 1000), 'lgbm_clean_full')]:
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                       model_path=f'{LGBM_DIR}/{sub}')
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_session_info=esi)
    print('  %-32s: %.4f' % (name, ndcg20(reranked)))
    report_by_turn(reranked, name)   # turn-1 = Blind proxy; gate on it
print('\n  full > 2k -> data starvation was real; scale further / ship full.')
print('  full ~= 2k -> not a data problem; need better features, not more rows.')


In [ ]:
# 12a) TIER-2 RETRAIN — relevance features (#4.1) + holdout early-stop + bagging (#4.3).
# Builds features WITH qwen_meta_cos + bm25_score over the full train split, drops
# the leaked cols (cfbpr_score, sasrec_rank_inv) like Stage 11, then trains with a
# leak-free temporal-holdout early stop + 5-seed bagging + drop-all-negative.
# Requires cell 1/3 (CACHE_DIR, ITEM_DB). ~1-2h (the build dominates). Run once.
import os, pandas as pd
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRAIN_RELEV = f'{LGBM_DIR}/lgbm_train_relev.parquet'
TRAIN_CL    = f'{LGBM_DIR}/lgbm_train_relev_clean.parquet'

# (1) temporal holdout ids (leak-free early-stop / selection set)
!cd /content/recsys2026 && python -u scripts/carve_temporal_selection_set.py \
    --frac 0.15 --out data/temporal_selection_split.json

# (2) build features WITH relevance over the FULL train split (force rebuild)
if os.path.exists(TRAIN_RELEV): os.remove(TRAIN_RELEV)
!cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
    --n-sessions 15199 --topk 100 --seed 42 \
    --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
    --with-relevance --cache-dir {CACHE_DIR} --out {TRAIN_RELEV}

# (3) drop the leaked model-derived cols (keep qwen_meta_cos + bm25_score)
df = pd.read_parquet(TRAIN_RELEV)
df.drop(columns=[c for c in ['cfbpr_score', 'sasrec_rank_inv'] if c in df.columns]).to_parquet(TRAIN_CL, index=False)
print('[retrain] clean parquet cols:', list(df.columns))

# (4) train: leak-free holdout early-stop + 5-seed bagging + drop-all-negative
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} \
    --holdout-ids /content/recsys2026/data/temporal_selection_split.json \
    --drop-all-negative --n-bag 5 \
    --output-dir {LGBM_DIR}/lgbm_relev


In [ ]:
# 12b) SCORE lgbm_relev on dev (turn-stratified) — injects the relevance features
# at scoring so they are actually exercised (parity with serve). Compares turn-1
# nDCG@20 vs recall-only and the references. Requires cell 4 + 4-strat + 12a.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.rerankers.relevance_scorer import RelevanceScorer
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.crs_baseline import build_sasrec_extra_features

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=f'{LGBM_DIR}/lgbm_relev')

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)

# per-candidate extra features: sasrec_rank + n_channels_hit, then merge relevance
efpc = build_sasrec_extra_features(per_sub, labels, fused100) or [[{} for _ in c] for c in fused100]
rel = RelevanceScorer(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR)
rel_feats = rel.feats_for_batch(queries, fused100)
for qi in range(len(fused100)):
    for ci in range(len(fused100[qi])):
        efpc[qi][ci].update(rel_feats[qi][ci])

esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                     goal_categories=goal_categories, goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_session_info=esi, extra_features_per_candidate=efpc)

def _ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)
print('recall-only baseline   :', round(_ndcg20([c[:20] for c in fused100]), 4))
print('references             : 2k=0.1712(turn1)  recall-only=0.1498(overall)')
report_by_turn(reranked, 'lgbm_relev (#4.1 relevance + #4.3 bagging + holdout)')


In [ ]:
# 12a-200) WIDER-POOL + ROUTING retrain (combined). Build features at topk=200
# WITH relevance features AND segment routing -> drop leaked cols -> train
# (holdout early-stop + 5-bag). topk=200 captures ~75% of the +0.072 recall
# headroom (recall@200=0.560 vs @100=0.506) and survives the session (~1.5-2.5h).
# Requires cell 1/3. Run once. (Set --topk 300 + Blackwell for the full version.)
import os, pandas as pd
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRAIN    = f'{LGBM_DIR}/lgbm_train_relev200.parquet'
TRAIN_CL = f'{LGBM_DIR}/lgbm_train_relev200_clean.parquet'

!cd /content/recsys2026 && python -u scripts/carve_temporal_selection_set.py \
    --frac 0.15 --out data/temporal_selection_split.json

if os.path.exists(TRAIN): os.remove(TRAIN)
!cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
    --n-sessions 15199 --topk 200 --seed 42 \
    --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
    --with-relevance --use-segment-routing \
    --cache-dir {CACHE_DIR} --out {TRAIN}

df = pd.read_parquet(TRAIN)
df.drop(columns=[c for c in ['cfbpr_score', 'sasrec_rank_inv'] if c in df.columns]).to_parquet(TRAIN_CL, index=False)
print('[retrain200] clean cols:', list(df.columns))

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRAIN_CL} \
    --holdout-ids /content/recsys2026/data/temporal_selection_split.json \
    --drop-all-negative --n-bag 5 \
    --output-dir {LGBM_DIR}/lgbm_relev200


In [ ]:
# 12b-200) SCORE lgbm_relev200 on dev — ROUTED 200-pool (matches build + serve).
# Builds a routed union, reuses its per-sub rankings, fuses with segment routing to
# top-200, injects relevance feats, reranks -> top-20. Turn-stratified; OVERALL is
# the Blind proxy. Requires cell 4 (queries/golds/turn_numbers/ctx/user_ids/played/
# goal_*) + 4-strat (report_by_turn) + 12a-200.
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.rerankers.relevance_scorer import RelevanceScorer
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.crs_baseline import build_sasrec_extra_features

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=f'{LGBM_DIR}/lgbm_relev200')

# Routed union: per-sub rankings are weight-independent; the routing lives in the
# fusion weights stored on sas_r.subs (cold/warm), matching the build exactly.
sas_r = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
        extra_config={'use_sasrec': True, 'w_sasrec': 1.0, 'use_segment_routing': True})
per_sub, labels = sas_r.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
cold_w = [s.get('cold_weight', s['weight']) for s in sas_r.subs]
warm_w = [s.get('warm_weight', s['weight']) for s in sas_r.subs]
is_warm = [bool((c or {}).get('history_tids')) for c in ctx]
fused200 = RRF_MODEL.fuse_per_sub_segmented(per_sub, cold_w, warm_w, sas_r.k, 200, is_warm)

efpc = build_sasrec_extra_features(per_sub, labels, fused200) or [[{} for _ in c] for c in fused200]
rel = RelevanceScorer(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR)
rel_feats = rel.feats_for_batch(queries, fused200)
for qi in range(len(fused200)):
    for ci in range(len(fused200[qi])):
        efpc[qi][ci].update(rel_feats[qi][ci])

esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
reranked = rr.rerank(queries, fused200, topk=20, user_ids=user_ids,
                     goal_categories=goal_categories, goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_session_info=esi, extra_features_per_candidate=efpc)

def _ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)
print('recall-only@200 routed (fused top-20):', round(_ndcg20([c[:20] for c in fused200]), 4))
print('references: 100-pool lgbm_relev overall=0.1652 | recall-only=0.1498')
report_by_turn(reranked, 'lgbm_relev200 (200-pool + relevance + routing + bagging)')


In [ ]:
# 12b-tt) DEV GATE for the two-tower-aware reranker (nb81 TRAIN_RERANKER=True -> lgbm_relev_tt).
# Scores it on dev with the SAME pool it trained on: union (SASRec + TWO-TOWER) -> top-{TOPK}
# -> relevance feats -> rerank -> top-20. OVERALL is the Blind proxy. No segment routing / no
# propose-ground here -> matches the nb81 cell-2 build (pg is not in the reranker's features).
# GATE: only submit the nb81 retrain to Blind if OVERALL beats 0.1652 by > 0.01.
# Requires cell 4 (queries/golds/turn_numbers/ctx/user_ids/played/goal_*) + 4-strat (report_by_turn)
# + the nb81 TRAIN_RERANKER=True run (writes {CACHE_DIR}/retrieval_v2/lgbm/lgbm_relev_tt).
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.rerankers.relevance_scorer import RelevanceScorer
from mcrs.retrieval_modules.rrf import RRF_MODEL
from mcrs.crs_baseline import build_sasrec_extra_features

TOPK = 100   # MUST equal the --topk used in the nb81 cell-2 build (RETRIEVAL_TOPK)
LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=f'{LGBM_DIR}/lgbm_relev_tt')

# Union WITH the two-tower channel (matches the reranker's training pool). Per-sub rankings
# are weight-independent; with no routing, cold/warm weights fall back to the fixed weight.
sas_r = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
        extra_config={'use_sasrec': True, 'w_sasrec': 1.0,
                      'use_two_tower': True, 'w_two_tower': 0.7})
per_sub, labels = sas_r.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
cold_w = [s.get('cold_weight', s['weight']) for s in sas_r.subs]
warm_w = [s.get('warm_weight', s['weight']) for s in sas_r.subs]
is_warm = [bool((c or {}).get('history_tids')) for c in ctx]
fused = RRF_MODEL.fuse_per_sub_segmented(per_sub, cold_w, warm_w, sas_r.k, TOPK, is_warm)

efpc = build_sasrec_extra_features(per_sub, labels, fused) or [[{} for _ in c] for c in fused]
rel = RelevanceScorer(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR)
rel_feats = rel.feats_for_batch(queries, fused)
for qi in range(len(fused)):
    for ci in range(len(fused[qi])):
        efpc[qi][ci].update(rel_feats[qi][ci])

esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                     goal_categories=goal_categories, goal_specificities=goal_specificities,
                     user_profiles_raw=[None] * len(queries),
                     extra_session_info=esi, extra_features_per_candidate=efpc)

def _ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)
ov = _ndcg20(reranked)
print('recall-only (fused top-20)        :', round(_ndcg20([c[:20] for c in fused]), 4))
print('references: lgbm_relev=0.1652 (submitted) | lgbm_relev200=0.1674')
print(f'lgbm_relev_tt OVERALL dev nDCG@20 = {round(ov, 4)}')
delta = ov - 0.1652
verdict = ('SUBMIT — beats 0.1652 by >0.01' if delta > 0.01
           else f'DO NOT SUBMIT — only +{delta:.4f} over 0.1652 (below the >0.01 bar)'
           if delta > 0 else f'DO NOT SUBMIT — {delta:.4f} vs 0.1652 (regressed)')
print('GATE:', verdict)
report_by_turn(reranked, 'lgbm_relev_tt (two-tower + relevance, dev gate)')


In [ ]:
# 12c-train) FINE-TUNE the in-pool SASRec (SASRec_Improved_Plan.md). Warm-start
# sasrec_v1 -> rank the gold within the SASRec-free pool (bm25+dense+same_artist,
# top-100), GOAL-FUL context. Writes sasrec_inpool_v1. SMOKE first (validates
# shapes/retrieval on 300 sessions / 1 epoch), THEN full. Requires cell 1 + cell 3
# (CACHE_DIR + sasrec_v1 present). Gate the result with the next cell (# 12c-inpool).
SMOKE = True   # True -> --n-sessions 300 --epochs 1 (quick check); False -> full fine-tune
_args = '--n-sessions 300 --epochs 1' if SMOKE else '--epochs 3'
!cd /content/recsys2026 && python -u scripts/train_sasrec_inpool.py \
    --cache-dir {CACHE_DIR} --warm-start sasrec_v1 --out sasrec_inpool_v1 \
    --topk 100 {_args}
if SMOKE:
    print('\n*** SMOKE done. If shapes + the gold-in-pool %% look sane, set SMOKE=False and re-run for the full fine-tune, then run # 12c-inpool. ***')


In [ ]:
# 12c-inpool) DEV GATE for the in-pool contrastive SASRec ranker (SASRec_Improved_Plan.md).
# Ranks the SERVE pool (union+SASRec, top-100) by the fine-tuned sasrec_inpool score
# with GOAL-FUL context (3-way parity), computes nDCG@20 on dev(=test), compares to
# lgbm_relev=0.1652 and recall-only=0.1498. Requires cell 1 (CACHE_DIR/ITEM_DB/CORPUS)
# + scripts/train_sasrec_inpool.py having written sasrec_inpool_v1. Self-contained.
import math, numpy as np, pandas as pd, torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import (
    SasrecModel, build_sasrec_context, prior_turns, build_user_dialog)
from mcrs.db_item import MusicCatalogDB

INPOOL_DIR = 'sasrec_inpool_v1'   # the model written by train_sasrec_inpool.py
TOPK = 100
dev_ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
item_db = MusicCatalogDB(ITEM_DB, ['all_tracks'], CORPUS)

# Build dev samples: union query (raw_with_goal), goal-ful SASRec context, played, gold.
from mcrs.crs_baseline import build_retrieval_query
q, ctx_txt, played, udlg, golds = [], [], [], [], []
for sess in dev_ds:
    df = pd.DataFrame(sess['conversations'])
    goal = ((sess.get('conversation_goal') or {}).get('listener_goal') or '').strip()
    up = sess.get('user_profile') or {}
    for _, m in df[df['role'] == 'music'].iterrows():
        tn = int(m['turn_number']); prior = prior_turns(df, tn).to_dict('records')
        sm = [{'role': ('assistant' if t['role'] == 'music' else t['role']),
               'content': (item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content'])}
              for t in prior]
        q.append(build_retrieval_query(sm, mode='raw_with_goal', goal_text=goal, user_profile=up))
        ctx_txt.append(build_sasrec_context(prior, goal_text=goal))
        udlg.append(build_user_dialog(prior))
        played.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        golds.append(m['content'])
print(f'[inpool-gate] {len(q)} dev (session,turn) samples')

# Serve pool = union WITH SASRec (matches config 194 serve), top-100.
union = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                              extra_config={'use_sasrec': True, 'w_sasrec': 1.0})
bctx = [{'history_tids': p, 'user_dialog': d} for p, d in zip(played, udlg)]
pools = []
for i in range(0, len(q), 256):
    pools.extend(union.batch_text_to_item_retrieval(q[i:i+256], topk=TOPK, batch_context=bctx[i:i+256]))

# Load the in-pool model + catalog feats.
dev_t = 'cuda' if torch.cuda.is_available() else 'cpu'
ck = torch.load(f'{CACHE_DIR}/retrieval_v2/sasrec/{INPOOL_DIR}/sasrec.pt', map_location='cpu', weights_only=False)
model = SasrecModel(**ck['model_kwargs']).to(dev_t).eval()
model.load_state_dict(ck['state_dict'])
feats_t = ck['item_feats'].to(dev_t); track_ids = ck['track_ids']
tid_to_idx = {t: i for i, t in enumerate(track_ids)}

# Encode goal-ful contexts (bge, normalize_embeddings=True to match train_sasrec_inpool /
# encode_dialogs_cached parity), build played sequences, score each pool.
st = SentenceTransformer('BAAI/bge-base-en-v1.5', device=dev_t)
# Truncation parity (train_sasrec_inpool / warm-start / serve factory): 512-cap,
# truncate LEFT so the most-recent turns + the trailing 'goal: ...' line survive.
# The tokenizer DEFAULT is 'right', which silently drops the goal + latest turns
# on >512-token contexts (5.4% of dev turns).
st.max_seq_length = 512
st.tokenizer.truncation_side = 'left'
ctx_emb = torch.as_tensor(st.encode(ctx_txt, batch_size=256, show_progress_bar=True,
                                    convert_to_numpy=True, normalize_embeddings=True), dtype=torch.float32, device=dev_t)
L = model.max_len; item_in = feats_t.shape[1]

def _ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g: s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

reranked = []
with torch.no_grad():
    for i in range(0, len(q), 256):
        bq = range(i, min(i + 256, len(q)))
        B = len(bq)
        pl = torch.zeros(B, L, item_in, device=dev_t); ln = torch.zeros(B, dtype=torch.long, device=dev_t)
        for r, j in enumerate(bq):
            idxs = [tid_to_idx[t] for t in played[j] if t in tid_to_idx][-L:]
            if idxs: pl[r, :len(idxs)] = feats_t[idxs]; ln[r] = len(idxs)
        state = torch.nn.functional.normalize(model.encode(ctx_emb[i:i+B], pl, ln), dim=-1)
        for r, j in enumerate(bq):
            cand = [t for t in pools[j] if t in tid_to_idx]
            if not cand: reranked.append(pools[j][:20]); continue
            cm = torch.nn.functional.normalize(model.item_fusion(feats_t[[tid_to_idx[t] for t in cand]]), dim=-1)
            sc = (cm @ state[r]).cpu().numpy()
            reranked.append([cand[k] for k in np.argsort(-sc)[:20]])

ov = _ndcg20(reranked)
print('references: lgbm_relev=0.1652 | recall-only=0.1498')
print(f'sasrec_inpool (alone) OVERALL dev nDCG@20 = {round(ov, 4)}')
delta = ov - 0.1652
print('GATE:', 'SUBMIT-worthy (>+0.01 over 0.1652)' if delta > 0.01
      else f'below bar (+{delta:.4f} vs 0.1652)' if delta > 0 else f'regressed ({delta:.4f})')
print('NEXT if promising: 3-way A/B vs LGBM-alone and LGBM+inpool-score-as-feature (plan 10.2).')


In [ ]:
# 12d-llm-listwise) GATE for Track A: LLM listwise reranker (plan: drastically
# improve nDCG@20). Ranks the config-194 serve pool (union+SASRec top-100, = cs
# from cell 4) with Gemini reasoning over real music knowledge. Reports OVERALL +
# TURN-1 nDCG@20 vs lgbm_relev=0.1652 / recall-only=0.1498, the pool recall@100
# (hard ceiling), and two failure-mode diagnostics: valid-index fraction (<0.7 =>
# mostly passthrough) and pop-skew (LLM top-20 popularity vs gold). Run
# TURN1_ONLY=True first (~$0.25). Requires cell 1 (ITEM_DB/CORPUS/CACHE_DIR) +
# cell 4 (queries, golds, played, user_ids, turn_numbers, goal_categories,
# goal_specificities, cs, item_db). Needs GEMINI_API_KEY / GOOGLE_API_KEY.
import numpy as np, pandas as pd
from datasets import load_dataset
from mcrs.eval_ndcg import ndcg_by_turn, format_by_turn
from mcrs.rerankers.llm_listwise_rerank import LLMListwiseReranker

TURN1_ONLY = True               # gate on the Blind proxy first (cheap); then False
K = 50                          # candidates shown to the LLM (plan default)
GEMINI_MODEL = 'gemini-2.5-flash-lite'

# user_profile per (session,turn), aligned to cell-4 order (cell 4 doesn't keep it).
_dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')
user_profiles = []
for sess in _dev:
    df = pd.DataFrame(sess['conversations']); up = sess.get('user_profile') or {}
    user_profiles += [up] * int((df['role'] == 'music').sum())
assert len(user_profiles) == len(queries), (len(user_profiles), len(queries))

pools = cs                      # union+SASRec top-100 (config-194 serve pool)
sel = [i for i in range(len(queries)) if (turn_numbers[i] == 1 or not TURN1_ONLY)]
print(f'[12d] ranking {len(sel)} turns (TURN1_ONLY={TURN1_ONLY}) K={K} model={GEMINI_MODEL}')

rr = LLMListwiseReranker(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                         model_path=GEMINI_MODEL, k=K)
reranked = rr.rerank([queries[i] for i in sel], [pools[i] for i in sel], topk=20,
                     user_ids=[user_ids[i] for i in sel],
                     goal_categories=[goal_categories[i] for i in sel],
                     goal_specificities=[goal_specificities[i] for i in sel],
                     user_profiles_raw=[user_profiles[i] for i in sel])

g = [golds[i] for i in sel]; tn = [turn_numbers[i] for i in sel]
rep = ndcg_by_turn(reranked, g, tn, k=20)
print('references: lgbm_relev=0.1652 | recall-only=0.1498')
print(format_by_turn(rep, f'llm_listwise k={K}'))

pr100 = float(np.mean([g[j] in pools[sel[j]][:100] for j in range(len(sel))]))
pr20  = float(np.mean([g[j] in pools[sel[j]][:20]  for j in range(len(sel))]))
rr20  = float(np.mean([g[j] in reranked[j][:20]    for j in range(len(sel))]))
print(f'[ceiling] pool recall@100={pr100:.4f} recall@20={pr20:.4f} -> reranked recall@20={rr20:.4f}')

npar = rr.diagnostics['n_parsed']; hl = rr.diagnostics['head_len']
vif = float(np.mean([(npar[j] or 0) / max(1, hl[j]) for j in range(len(sel))]))
print(f'[diag] valid-index fraction = {vif:.3f}  (<0.70 => mostly passthrough)')

def _pop(tid):
    md = item_db.metadata_dict.get(tid) or {}; return float(md.get('popularity') or 0.0)
top_pop = float(np.mean([np.mean([_pop(t) for t in reranked[j][:20]]) for j in range(len(sel))]))
gold_pop = float(np.mean([_pop(x) for x in g]))
print(f'[diag] mean popularity: LLM top-20={top_pop:.1f} vs gold={gold_pop:.1f} '
      f'({"SKEW: LLM favors popular" if top_pop > gold_pop + 5 else "ok"})')

t1 = rep.get('turn1'); ov = rep['overall']
print(f'\nGATE (turn-1 = Blind proxy): turn1={t1} overall={ov:.4f}')
print('  SHIP-worthy' if (t1 is not None and t1 > 0.1652 and vif >= 0.7)
      else '  below bar / uninformative (see diagnostics)')
print('NEXT: if turn-1 clears -> TURN1_ONLY=False (full dev), then configs 198/199 Blind-A.')


## Stage 12 - query-signal A/B: does adding taste/intent crack the new-artist wall?

Schema audit found the retrieval query (cell 4) uses conversation turns ONLY -- it
withholds listener_goal (intent text) and preferred_musical_culture (taste). The
50% recall wall is 98.8% new-artist; session channels can't reach those, so the
ONLY path is content matching INTENT -- exactly what's withheld. This A/Bs the
query on union recall@100:
  A turns-only (reproduce 0.4989 baseline)
  B + listener_goal text
  C + listener_goal + preferred_musical_culture (+ preferred_language)
and reports new-artist rescues (golds A missed that B/C surface). Free on retrieval
(dense re-encodes the new query text once). If recall rises, this is a real recall
lever -> wire into _wrrf_union_v1 query construction + build_lgbm_features (parity).
Caveat: dense may already infer intent from conversation text; measure before believing.

In [ ]:
# Stage 12: query-signal A/B. Requires cell 1/3 (ITEM_DB/CORPUS/CACHE_DIR) and
# cell 4 (queries, user_ids, played, golds, item_db, build_user_dialog). Rebuilds
# the dev set from scratch so we can attach profile/goal to each query variant.
import numpy as np, pandas as pd
from datasets import load_dataset
from mcrs.retrieval_modules import load_retrieval_module
from mcrs.retrieval_modules.sasrec_model import build_user_dialog

dev = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='test')

q_turns, q_goal, q_full = [], [], []      # three query variants, aligned
g2, uids, played2, udlg = [], [], [], []
for sess in dev:
    df = pd.DataFrame(sess['conversations'])
    up = sess.get('user_profile') or {}
    cg = sess.get('conversation_goal') or {}
    goal_txt = (cg.get('listener_goal') or '').strip()
    cult = (up.get('preferred_musical_culture') or '').strip()
    lang = (up.get('preferred_language') or '').strip()
    for _, music in df[df['role'] == 'music'].iterrows():
        tn = int(music['turn_number'])
        prior = df[(df['turn_number'] < tn) |
                   ((df['turn_number'] == tn) & (df['role'] == 'user'))]
        lines = []
        for _, t in prior.iterrows():
            role = 'assistant' if t['role'] == 'music' else t['role']
            content = item_db.id_to_metadata(t['content']) if t['role'] == 'music' else t['content']
            lines.append(f'{role}: {content}')
        base = chr(10).join(lines)
        q_turns.append(base)
        q_goal.append(base + (f'{chr(10)}goal: {goal_txt}' if goal_txt else ''))
        extra = (f'{chr(10)}goal: {goal_txt}' if goal_txt else '')
        if cult: extra += f'{chr(10)}taste: {cult}'
        if lang: extra += f'{chr(10)}language: {lang}'
        q_full.append(base + extra)
        g2.append(music['content']); uids.append(sess.get('user_id'))
        played2.append(list(df[(df['role'] == 'music') & (df['turn_number'] < tn)]['content']))
        udlg.append(build_user_dialog(prior.to_dict('records')))
ctx2 = [{'history_tids': p, 'user_dialog': d} for p, d in zip(played2, udlg)]
print('[stage12] built', len(q_turns), 'turns x 3 query variants')

# union+SASRec, instruct dense (current best retrieval).
uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                            extra_config={'use_sasrec': True, 'w_sasrec': 1.0})

def recall_at(cands, k):
    return float(np.mean([1.0 if g in c[:k] else 0.0 for c, g in zip(cands, g2)]))

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {}); a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

print('\n=== Stage 12 query-signal A/B (union+SASRec recall@100, dev n=%d) ===' % len(g2))
res = {}
for name, qs in [('A turns-only', q_turns), ('B +goal', q_goal), ('C +goal+taste', q_full)]:
    cand = uni.batch_text_to_item_retrieval(qs, topk=100, user_ids=uids, batch_context=ctx2)
    res[name] = cand
    print('  %-16s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

# new-artist rescues: golds A missed @100 that C now surfaces
miss_A = [i for i in range(len(g2)) if g2[i] not in set(res['A turns-only'][i][:100])]
resc = [i for i in miss_A if g2[i] in set(res['C +goal+taste'][i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(g2[i]) is not None and artist_of(g2[i]) in {artist_of(t) for t in played2[i]}))
print('\n  A-missed golds @100: %d' % len(miss_A))
print('  rescued by C       : %d' % len(resc))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING --')
print('  C/B recall > A -> withheld taste/intent IS a recall lever; wire into the query.')
print('  C/B ~= A       -> dense already captures intent from turns; query text not the gap.')


## Stage 13 - lyrics channel A/B: does a 2nd content view crack the new-artist wall?

A1 (roadmap 2026-05-30): add dense_lyrics_qwen3_instruct (precomputed lyrics-qwen3
embeddings) as a union channel. Different content view than metadata-dense, so it
can surface new-artist golds the metadata/lexical/session channels miss (the 50%
wall, 98.8% new-artist). Opt-in via use_lyrics. This A/Bs union+SASRec recall@100
WITHOUT vs WITH lyrics (sweeping its weight), and reports new-artist rescues.
Uses the A2-fixed query (cell 4, incl. listener_goal). Free-ish: dense re-encodes
queries once for the lyrics column; track embeddings precomputed.

In [ ]:
# Stage 13: lyrics channel A/B. Requires cell 4 (queries, user_ids, ctx, golds,
# played, item_db, recall_at). Baseline = union+SASRec (the shipped config 194).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    m = item_db.metadata_dict.get(tid, {}); a = m.get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no lyrics)', dict(base_cfg)),
        ('B + lyrics w=0.4',          dict(base_cfg, use_lyrics=True, w_lyrics=0.4)),
        ('C + lyrics w=0.7',          dict(base_cfg, use_lyrics=True, w_lyrics=0.7))]

print('=== Stage 13 lyrics channel A/B (union+SASRec recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    print('  %-28s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

# new-artist rescues: golds A missed @100 that the best lyrics run now surfaces
A = cand_by['A union+SASRec (no lyrics)']
best = max([k for k in cand_by if k != 'A union+SASRec (no lyrics)'],
           key=lambda k: recall_at(cand_by[k], 100))
B = cand_by[best]
miss_A = [i for i in range(len(golds)) if golds[i] not in set(A[i][:100])]
resc = [i for i in miss_A if golds[i] in set(B[i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(golds[i]) is not None and artist_of(golds[i]) in {artist_of(t) for t in played[i]}))
print('\n  vs %s:' % best)
print('  A-missed golds @100: %d  | rescued: %d' % (len(miss_A), len(resc)))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING --')
print('  lyrics recall > A + new-artist rescues -> add use_lyrics to config 194/195, ship.')
print('  lyrics ~= A -> lyrics redundant with metadata/sasrec; drop it.')


## Stage 14 - train-on-300: does training the reranker on deeper candidates help?

Stage 9 widened the SERVE pool (100->500) with a reranker TRAINED on 100 -> recall
+0.07 but nDCG flat (reranker couldn't rank deep golds it never trained on). The
untested variant: TRAIN the reranker on a 300-candidate pool (deeper negatives +
deeper golds), then serve at 300. Cheaper than OOF (NO SASRec retrains; reuses
sasrec_v1). Compares dev nDCG@20:
  - clean_full (trained@100, served@100) = 0.1623  (shipped best)
  - clean_full (trained@100, served@300)            (the Stage 9 negative, re-confirmed)
  - clean_300  (trained@300, served@300)            (this experiment)
If clean_300@300 > 0.1623, training-on-deeper-pool converts latent recall -> ship it.

In [ ]:
# Stage 14: train-on-300. Requires cell 1/3 (CACHE_DIR/ITEM_DB/CORPUS) and cell 4
# (sas, queries, user_ids, ctx, played, golds, turn_numbers, goal_categories,
# goal_specificities). Idempotent: skips finished parquets.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TR300 = f'{LGBM_DIR}/lgbm_train_full300_sasrec.parquet'
VA300 = f'{LGBM_DIR}/lgbm_val_full300_sasrec.parquet'

# 1) Build features at topk=300 (same 15k/2k scale as clean_full, with sasrec channel).
if not os.path.exists(TR300):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15000 --topk 300 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TR300}
if not os.path.exists(VA300):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 300 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VA300}

# 2) Drop leaked columns -> clean feature set (same as clean_full).
TRC = f'{LGBM_DIR}/lgbm_train_clean300.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_clean300.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TR300, TRC), (VA300, VAC)]:
    df = pd.read_parquet(s)
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[stage14] built clean-300 parquets (dropped', LEAKED, ')')

# 3) Train lgbm_clean_300.
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} \
    --output-dir {LGBM_DIR}/lgbm_clean_300

# 4) Eval: 3 conditions. Fuse once at 300; slice to 100 for the trained@100 model.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused300 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 300)
fused100 = [f[:100] for f in fused300]
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

def rr_ndcg(sub, pool):
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        return None
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, pool, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_session_info=esi)
    return ndcg20(reranked)

print('\n=== Stage 14 train-on-300 DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only @20 (fused100)        :', round(ndcg20([r[:20] for r in fused100]), 4))
a = rr_ndcg('lgbm_clean_full', fused100)
b = rr_ndcg('lgbm_clean_full', fused300)
c = rr_ndcg('lgbm_clean_300',  fused300)
print('  clean_full  trained@100 served@100:', round(a, 4) if a else 'NA', '  <- shipped 0.1623')
print('  clean_full  trained@100 served@300:', round(b, 4) if b else 'NA', '  <- Stage 9 negative')
print('  clean_300   trained@300 served@300:', round(c, 4) if c else 'NA', '  <- this experiment')
print('\n-- READING --')
if a and c:
    print('  clean_300 %+.4f vs shipped clean_full' % (c - a))
    print('  > 0 -> training on a deeper pool converts latent recall; rebuild config with topk 300.')
    print('  <=0 -> deeper-pool training does not help; reranker is feature-limited, not pool-limited.')


## Stage 15 - RRF weight + k sweep (FREE, no retrieval): tune the union mix

The union weights were hand-set and never swept: bm25=1.0, dense=0.7, same_artist=1.0,
sasrec=1.0, k=60. But sasrec is the strongest channel (0.4339) yet weighted = bm25
(0.3957), and the weak instruct-dense (0.179 isolated) gets 0.7 (likely RRF noise).
fuse_per_sub is a PURE function: we pull per-sub rankings ONCE (batch_per_sub_rankings)
then grid-search weights + k in pure Python over the cached lists — zero GPU, minutes.
Reports recall@100 (and @20) per combo; baseline = current shipped weights. RRF score
scales linearly in weight, so we fix bm25=1.0 and sweep the others' RATIO + k.

In [ ]:
# Stage 15: RRF weight + k sweep. Requires cell 4 (sas, queries, user_ids, ctx, golds).
# Pure re-fusion over cached per-sub rankings — no retriever re-run.
from mcrs.retrieval_modules.rrf import RRF_MODEL

# 1) Pull per-channel rankings ONCE (the only expensive step; reuses dense cache).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
print('[stage15] channels:', labels)

def recall_at(cands, k):
    hit = 0
    for c, g in zip(cands, golds):
        if g in c[:k]:
            hit += 1
    return hit / len(golds)

def fuse(weight_by_label, k, topk=100):
    # align weights to per_sub order via labels
    w = [float(weight_by_label.get(lbl, 1.0)) for lbl in labels]
    return RRF_MODEL.fuse_per_sub(per_sub, w, k, topk)

# 2) Baseline = current shipped weights (bm25=1.0, dense=0.7, artist=1.0, sasrec=1.0, k=60).
BM, DE, AR, SA = 'bm25', 'dense_metadata_qwen3_instruct', 'same_artist', 'sasrec_seq'
base_w = {BM: 1.0, DE: 0.7, AR: 1.0, SA: 1.0}
base = fuse(base_w, 60)
base_r100 = recall_at(base, 100)
print('[stage15] BASELINE (shipped) recall@100=%.4f recall@20=%.4f' % (base_r100, recall_at(base, 20)))

# 3) Grid: fix bm25=1.0 (RRF is scale-invariant), sweep the rest + k.
grid_qwen   = [0.3, 0.5, 0.7, 1.0]
grid_artist = [0.7, 1.0, 1.5]
grid_sasrec = [1.0, 1.5, 2.0, 2.5]
grid_k      = [20, 40, 60]

results = []
for wq in grid_qwen:
    for wa in grid_artist:
        for ws in grid_sasrec:
            for k in grid_k:
                w = {BM: 1.0, DE: wq, AR: wa, SA: ws}
                fused = fuse(w, k)
                results.append((recall_at(fused, 100), recall_at(fused, 20), wq, wa, ws, k))

results.sort(key=lambda r: -r[0])
print('\n[stage15] top 10 combos by recall@100 (qwen/artist/sasrec/k):')
for r100, r20, wq, wa, ws, k in results[:10]:
    tag = '  *BEATS baseline' if r100 > base_r100 else ''
    print('  r@100=%.4f r@20=%.4f | qwen=%.1f artist=%.1f sasrec=%.1f k=%d%s'
          % (r100, r20, wq, wa, ws, k, tag))

best = results[0]
print('\n[stage15] BEST recall@100=%.4f vs baseline %.4f (delta %+.4f)'
      % (best[0], base_r100, best[0] - base_r100))
print('  winning weights: w_qwen=%.1f w_artist=%.1f w_sasrec=%.1f k=%d (bm25=1.0)'
      % (best[2], best[3], best[4], best[5]))
print('\n-- READING --')
print('  delta > ~0.005 -> wire winning weights into config extra_config (w_qwen/w_artist/w_sasrec)')
print('     NOTE: also rebuild LGBM features with the same weights for train/serve parity,')
print('     and the rerank fused order changes -> re-check nDCG (Stage 3/11) before shipping.')
print('  delta ~0 -> current hand-set weights were already near-optimal; move on.')


## Stage 16 - related-artist co-occurrence PROBE (no GPU, no new code): is Lever 3 worth building?

The ~50% recall wall is 98.8% NEW-ARTIST; same_artist can't reach them. Lever 3 =
a related-artist channel (cross-session artist co-occurrence). Before building a
retriever, this probe measures the UPPER BOUND: build an artist->co-occurring-artist
graph from the 15k TRAIN sessions, then for each dev gold the union MISSED @100,
check whether the gold's artist is reachable by expanding the session's seen artists
to their top-K co-occurring artists. The reachable fraction = the max a related-artist
channel could rescue. Pure counting, minutes, no model code. Decides whether to build.

In [ ]:
# Stage 16: related-artist co-occurrence probe. Requires cell 4 (cs, played, golds,
# item_db) + the train split. No GPU, no new retriever — pure counting.
from collections import Counter, defaultdict
from datasets import load_dataset
import pandas as pd

md_dict = item_db.metadata_dict
def artist_of(tid):
    a = md_dict.get(tid, {}).get('artist_name')
    if isinstance(a, list):
        a = a[0] if a else ''
    return str(a or '').strip().lower()

# 1) Build artist co-occurrence from TRAIN sessions (artists sharing a session).
print('[stage16] building artist co-occurrence from train...')
tr = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
cooc = defaultdict(Counter)
for sess in tr:
    df = pd.DataFrame(sess['conversations'])
    arts = []
    for tid in df[df['role'] == 'music']['content']:
        a = artist_of(tid)
        if a:
            arts.append(a)
    uniq = list(dict.fromkeys(arts))  # unique, order-preserving
    for a in uniq:
        for b in uniq:
            if a != b:
                cooc[a][b] += 1
print('[stage16] artists with co-occurrence:', len(cooc))

# 2) For each dev turn the union MISSED @100, is the gold artist reachable by
#    expanding the session's seen artists to top-K co-occurring artists?
def reachable_at(K):
    n_miss = 0; n_new = 0; n_reach = 0
    for i in range(len(golds)):
        if golds[i] in set(cs[i][:100]):
            continue  # union already got it
        n_miss += 1
        g_art = artist_of(golds[i])
        seen = {artist_of(t) for t in played[i]}
        seen.discard('')
        is_new = g_art != '' and g_art not in seen
        if not is_new:
            continue
        n_new += 1
        # expand: rank all artists co-occurring with any seen artist, by summed count
        exp = Counter()
        for s in seen:
            for b, c in cooc.get(s, {}).items():
                if b not in seen:
                    exp[b] += c
        topK = {a for a, _ in exp.most_common(K)}
        if g_art in topK:
            n_reach += 1
    return n_miss, n_new, n_reach

print('\n=== Stage 16 related-artist rescue ceiling (union-missed dev golds) ===')
m, nw, _ = reachable_at(10)
print('  union-missed @100        : %d' % m)
print('  of which new-artist      : %d (%.1f%%)' % (nw, 100.0 * nw / max(1, m)))
print('  reachable via co-occurrence expansion (of the new-artist misses):')
for K in (10, 50, 100, 200, 500):
    _, nw2, nr = reachable_at(K)
    print('    top-%-4d co-occ artists : %d (%.1f%% of new-artist misses, %.1f%% of all misses)'
          % (K, nr, 100.0 * nr / max(1, nw2), 100.0 * nr / max(1, m)))

print('\n-- READING --')
print('  high reachable %% (e.g. >15-20%% of misses at K<=200) -> related-artist channel is')
print('     worth building (Lever 3): co-occurrence genuinely reaches the wall artists.')
print('  low reachable %% -> wall artists are cold (no train co-occurrence); the channel')
print('     would not help -> skip Lever 3, the wall needs content/intent not artist-CF.')


## Stage 17 - Lever 2: n_channels_hit reranker feature (cross-channel agreement)

Stage 10 found the reranker feature-limited (no strong relevance signal). Lever 2
adds n_channels_hit = how many union channels surfaced each candidate (golds tend
to be multiply-surfaced) — a free cross-channel-agreement relevance feature, now
plumbed train+serve (WRRFRunner.run + build_sasrec_extra_features, guarded for
parity). This rebuilds full features (which now include the column), trains
lgbm_clean_full_nch (clean feature set + n_channels_hit), and compares dev nDCG@20
vs clean_full (0.1623). COST: one 15k feature rebuild (~40min, the column needs
fresh parquets) + CPU train. Idempotent.

In [ ]:
# Stage 17: n_channels_hit reranker feature. Requires cell 1/3 + cell 4 (sas,
# queries, user_ids, ctx, golds, played, turn_numbers, goal_categories,
# goal_specificities). Idempotent (skips finished parquets).
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
# Fresh full-scale parquets that include n_channels_hit (the pre-Lever-2 parquets
# lack the column). Distinct filenames so we don't clobber lgbm_*_full_sasrec.
TR = f'{LGBM_DIR}/lgbm_train_full_nch.parquet'
VA = f'{LGBM_DIR}/lgbm_val_full_nch.parquet'
if not os.path.exists(TR):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15000 --topk 100 --seed 42 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {TR}
if not os.path.exists(VA):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 7 \
        --use-sasrec --w-sasrec 1.0 --sasrec-model-dir sasrec_v1 \
        --cache-dir {CACHE_DIR} --out {VA}

# Clean feature set (drop leaked cols) but KEEP n_channels_hit.
TRC = f'{LGBM_DIR}/lgbm_train_clean_nch.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_clean_nch.parquet'
LEAKED = ['cfbpr_score', 'sasrec_rank_inv']
for s, d in [(TR, TRC), (VA, VAC)]:
    df = pd.read_parquet(s)
    assert 'n_channels_hit' in df.columns, 'rebuild missing n_channels_hit — pull latest code'
    df.drop(columns=[c for c in LEAKED if c in df.columns]).to_parquet(d, index=False)
print('[stage17] clean+nch parquets ready; n_channels_hit present')

!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} \
    --output-dir {LGBM_DIR}/lgbm_clean_full_nch

# Eval vs clean_full. Feed efpc with BOTH sasrec_rank (ignored by clean models)
# and n_channels_hit, computed from per_sub exactly like build_sasrec_extra_features.
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    hit = {}
    for s in range(len(per_sub)):
        for t in per_sub[s][qi]:
            hit[t] = hit.get(t, 0) + 1
    efpc.append([{'sasrec_rank': rm.get(tid, 10000),
                  'n_channels_hit': hit.get(tid, 1)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 17 n_channels_hit DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('clean_full      (no nch)', 'lgbm_clean_full'),
                  ('clean_full_nch  (+nch)  ', 'lgbm_clean_full_nch')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  ' + name + ' :', round(ndcg20(reranked), 4))
print('\n-- READING: clean_full_nch > 0.1623 -> n_channels_hit helps; ship it (config 195). --')


## Stage 18 - undertrained reranker: more boosting rounds (n_estimators)

Stage 11/14 LGBM logs show "Did not meet early stopping. Best iteration is [1000]"
with val nDCG STILL RISING monotonically at the cap — the reranker is UNDERTRAINED,
truncated by n_estimators=1000, not converged. This retrains clean_full on the SAME
parquets (no rebuild) at higher caps and compares dev nDCG@20 vs 0.1637.
CAVEAT: the rising curve is INTERNAL val (optimistic, conditional-on-present);
more rounds may just overfit. Must judge on DEV, not internal val. Cheap (CPU,
minutes — just more boosting on existing features).

In [ ]:
# Stage 18: more boosting rounds. Requires cell 1/3 + cell 4. Reuses the Stage 11
# clean full parquets (no feature rebuild). Idempotent per output dir.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

LGBM_DIR = f'{CACHE_DIR}/retrieval_v2/lgbm'
TRC = f'{LGBM_DIR}/lgbm_train_full_clean.parquet'
VAC = f'{LGBM_DIR}/lgbm_val_full_clean.parquet'
assert os.path.exists(TRC), 'Run Stage 11 first (lgbm_train_full_clean.parquet missing)'

# Train at increasing round caps. lr stays 0.05 (hardcoded in train_lgbm_ranker);
# early-stopping=50 still active, so a cap that converges will stop on its own.
for n_est in (2000, 4000):
    out = f'{LGBM_DIR}/lgbm_clean_full_n{n_est}'
    if os.path.exists(f'{out}/booster.txt'):
        print(f'[stage18] {out} present, skip'); continue
    !cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
        --train-features {TRC} --val-features {VAC} \
        --n-estimators {n_est} --early-stopping 100 \
        --output-dir {out}

# Eval on dev vs shipped clean_full (n=1000).
per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
weights = [s['weight'] for s in sas.subs]
fused100 = RRF_MODEL.fuse_per_sub(per_sub, weights, sas.k, 100)
sidx = labels.index('sasrec_seq')
efpc = []
for qi, cands in enumerate(fused100):
    rm = {tid: r + 1 for r, tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank': rm.get(tid, 10000)} for tid in cands])
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]

def ndcg20(ranked):
    s = 0.0
    for r, g in zip(ranked, golds):
        for pos, tid in enumerate(r[:20]):
            if tid == g:
                s += 1.0 / math.log2(pos + 2); break
    return s / len(golds)

print('\n=== Stage 18 more-rounds DEV nDCG@20 (n=%d) ===' % len(golds))
print('  recall-only:', round(ndcg20([r[:20] for r in fused100]), 4))
for name, sub in [('clean_full  n=1000 (shipped)', 'lgbm_clean_full'),
                  ('clean_full  n=2000', 'lgbm_clean_full_n2000'),
                  ('clean_full  n=4000', 'lgbm_clean_full_n4000')]:
    mp = f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'):
        print('  ' + name + ' : (missing)'); continue
    rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, model_path=mp)
    reranked = rr.rerank(queries, fused100, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories,
                         goal_specificities=goal_specificities,
                         user_profiles_raw=[None] * len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  ' + name + ' :', round(ndcg20(reranked), 4))
print('\n-- READING --')
print('  dev nDCG rises with rounds -> was undertrained; ship the best n_estimators.')
print('  dev nDCG flat/falls -> internal-val rise was overfitting; keep n=1000.')


## Stage 19 - Lever 4: cf-bpr union channel A/B (orthogonal user-taste signal)

cf-bpr (user x item affinity) is built + factory-wired but NOT in the union — only
an LGBM feature. As a recall channel it's ORTHOGONAL to session (same_artist/sasrec)
and content (bm25/dense): for warm users it can surface popular tracks by NEW artists
their latent taste likes. Opt-in use_cfbpr (default w=0.25; ~43% warm, cold -> empty
list, RRF falls back). A/Bs union+SASRec recall@100 without vs with cf-bpr, with a
warm/cold split (the lever only helps warm users; must not regress cold). Free-ish:
cf-bpr is numpy-only, no GPU encode.

In [ ]:
# Stage 19: cf-bpr union channel A/B. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at, user_dialogs).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no cfbpr)', dict(base_cfg)),
        ('B + cfbpr w=0.25',         dict(base_cfg, use_cfbpr=True, w_cfbpr=0.25)),
        ('C + cfbpr w=0.5',          dict(base_cfg, use_cfbpr=True, w_cfbpr=0.5))]

# warm = user has a cf-bpr embedding. Load the cf_bpr retriever once to get the set.
cfr = load_retrieval_module('cf_bpr', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config={})
warm = set(cfr.user_embs.keys())
warm_mask = [uid in warm for uid in user_ids]
n_warm = sum(warm_mask)
print('[stage19] warm users: %d/%d (%.1f%%)' % (n_warm, len(user_ids), 100.0*n_warm/len(user_ids)))

def recall_split(cands, k):
    def r(mask):
        idx = [i for i in range(len(golds)) if mask[i]]
        if not idx: return float('nan')
        return sum(1.0 for i in idx if golds[i] in set(cands[i][:k])) / len(idx)
    return recall_at(cands, k), r(warm_mask), r([not m for m in warm_mask])

print('\n=== Stage 19 cf-bpr union A/B (recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    overall, warm_r, cold_r = recall_split(cand, 100)
    print('  %-26s overall=%.4f  warm=%.4f  cold=%.4f' % (name, overall, warm_r, cold_r))

print('\n-- READING --')
print('  warm recall rises AND cold recall not down -> cf-bpr is an orthogonal win; ship use_cfbpr.')
print('  warm ~flat or cold regresses -> cf-bpr adds RRF noise; drop it.')


## Stage 20 - does the Stage 15 weight win convert to nDCG?

Stage 15 found w_artist=1.5, w_sasrec=1.5 (bm25=1.0, qwen=0.7, k=60) lifts
recall@100 0.5061 -> 0.5154. But recall up != nDCG up (Stages 8/9/14). This
reranks BOTH the baseline-weight and new-weight fused pools with the SAME shipped
clean_full reranker and compares dev nDCG@20. APPROXIMATE: clean_full was trained
on old-weight features, so this under-credits the new weights slightly. If nDCG
holds/rises even approximately -> worth a full feature rebuild at the new weights.

In [ ]:
# Stage 20: reweighted-union nDCG check. Requires cell 4 (sas, queries, user_ids,
# ctx, golds, played, turn_numbers, goal_categories, goal_specificities).
import math
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL

per_sub, labels = sas.batch_per_sub_rankings(queries, user_ids=user_ids, batch_context=ctx)
def wvec(by_label):
    return [float(by_label.get(l, 1.0)) for l in labels]
BM, DE, AR, SA = 'bm25', 'dense_metadata_qwen3_instruct', 'same_artist', 'sasrec_seq'
base_w = {BM:1.0, DE:0.7, AR:1.0, SA:1.0}
new_w  = {BM:1.0, DE:0.7, AR:1.5, SA:1.5}

def recall_at(c, k): return sum(1.0 for x,g in zip(c,golds) if g in set(x[:k]))/len(golds)
def ndcg20(ranked):
    s=0.0
    for r,g in zip(ranked,golds):
        for pos,tid in enumerate(r[:20]):
            if tid==g: s+=1.0/math.log2(pos+2); break
    return s/len(golds)

rr = LGBM_RERANKER(ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR,
                   model_path=f'{CACHE_DIR}/retrieval_v2/lgbm/lgbm_clean_full')
esi = [{'played_tids': played[i], 'turn_number': turn_numbers[i],
        'prior_track_count': len(played[i])} for i in range(len(queries))]
sidx = labels.index('sasrec_seq')

print('=== Stage 20 reweighted-union nDCG@20 (clean_full reranker, dev n=%d) ===' % len(golds))
for name, w in [('baseline weights', base_w), ('NEW weights (a=1.5,s=1.5)', new_w)]:
    fused = RRF_MODEL.fuse_per_sub(per_sub, wvec(w), 60, 100)
    efpc = []
    for qi, cands in enumerate(fused):
        rm = {tid: r+1 for r,tid in enumerate(per_sub[sidx][qi])}
        efpc.append([{'sasrec_rank': rm.get(tid,10000)} for tid in cands])
    reranked = rr.rerank(queries, fused, topk=20, user_ids=user_ids,
                         goal_categories=goal_categories, goal_specificities=goal_specificities,
                         user_profiles_raw=[None]*len(queries),
                         extra_features_per_candidate=efpc, extra_session_info=esi)
    print('  %-26s recall@100=%.4f  nDCG@20=%.4f' % (name, recall_at(fused,100), ndcg20(reranked)))
print('\n-- READING: NEW nDCG >= baseline -> rebuild features at new weights + ship config 195. --')
print('   (approximate: clean_full trained on old-weight features, so this under-credits NEW.)')


## Stage 21 - Lever 3: related-artist channel A/B (attacks the new-artist wall)

Stage 16 probe: ~29% of union-missed golds reachable via artist co-occurrence at
top-100 — the biggest recall ceiling of the session, and the ONLY lever aimed at
the ~96%-new-artist wall. This adds the RelatedArtistRetriever as an opt-in 5th
union channel (use_related_artist) and A/Bs union+SASRec recall@100 without vs with
it (weight sweep), with the new-artist-rescue breakdown. First run builds + caches
the co-occurrence map from train (~1-2 min, no GPU); later runs load the cache.

In [ ]:
# Stage 21: related-artist channel A/B. Requires cell 4 (queries, user_ids, ctx,
# golds, played, item_db, recall_at).
from mcrs.retrieval_modules import load_retrieval_module

def artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no related)', dict(base_cfg)),
        ('B + related w=0.5',           dict(base_cfg, use_related_artist=True, w_related_artist=0.5)),
        ('C + related w=1.0',           dict(base_cfg, use_related_artist=True, w_related_artist=1.0)),
        ('D + related w=1.5',           dict(base_cfg, use_related_artist=True, w_related_artist=1.5))]

print('=== Stage 21 related-artist channel A/B (recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    print('  %-28s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

A = cand_by['A union+SASRec (no related)']
best = max([k for k in cand_by if k != 'A union+SASRec (no related)'], key=lambda k: recall_at(cand_by[k], 100))
B = cand_by[best]
miss_A = [i for i in range(len(golds)) if golds[i] not in set(A[i][:100])]
resc = [i for i in miss_A if golds[i] in set(B[i][:100])]
resc_new = sum(1 for i in resc
               if not (artist_of(golds[i]) is not None and artist_of(golds[i]) in {artist_of(t) for t in played[i]}))
print('\n  vs %s:' % best)
print('  A-missed @100: %d  | rescued: %d' % (len(miss_A), len(resc)))
if resc:
    print('    of which new-artist: %d (%.1f%%)' % (resc_new, 100.0*resc_new/len(resc)))
print('\n-- READING --')
print('  related recall > A with new-artist rescues -> the wall lever WORKS; add use_related_artist')
print('     to config 195, rebuild LGBM features at that union, re-check nDCG, ship.')
print('  related ~= A -> co-occurrence ranking too noisy at serve; tune weight / artist-topk.')


## Stage 22 - CLAP audio-similarity reranker feature (clap_session_sim)

CLAP audio only feeds SASRec item-feats today; never the reranker. clap_session_sim
= mean CLAP cosine of a candidate to the session's played tracks ("does it sound
like what was played") — a NEW audio relevance feature (features don't suffer RRF
noise). Rebuilds full features WITH --use-clap, drops leaked cols (keeps clap),
trains lgbm_clean_full_clap, compares dev nDCG@20 vs clean_full 0.1637.

In [ ]:
# Stage 22: CLAP feature. Requires cell 1/3 + cell 4. Idempotent.
import os, math
import pandas as pd
from mcrs.rerankers.lgbm_rerank import LGBM_RERANKER
from mcrs.retrieval_modules.rrf import RRF_MODEL
LGBM_DIR=f'{CACHE_DIR}/retrieval_v2/lgbm'
TR=f'{LGBM_DIR}/lgbm_train_full_clap.parquet'; VA=f'{LGBM_DIR}/lgbm_val_full_clap.parquet'
if not os.path.exists(TR):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 15000 --topk 100 --seed 42 --use-sasrec --w-sasrec 1.0 \
        --sasrec-model-dir sasrec_v1 --use-clap --cache-dir {CACHE_DIR} --out {TR}
if not os.path.exists(VA):
    !cd /content/recsys2026 && python -u scripts/build_lgbm_features.py \
        --n-sessions 2000 --topk 100 --seed 7 --use-sasrec --w-sasrec 1.0 \
        --sasrec-model-dir sasrec_v1 --use-clap --cache-dir {CACHE_DIR} --out {VA}
TRC=f'{LGBM_DIR}/lgbm_train_clean_clap.parquet'; VAC=f'{LGBM_DIR}/lgbm_val_clean_clap.parquet'
for s,d in [(TR,TRC),(VA,VAC)]:
    df=pd.read_parquet(s)
    assert 'clap_session_sim' in df.columns, 'rebuild missing clap col — pull latest'
    df.drop(columns=[c for c in ['cfbpr_score','sasrec_rank_inv'] if c in df.columns]).to_parquet(d,index=False)
print('[stage22] clean+clap parquets ready')
!cd /content/recsys2026 && python -u scripts/train_lgbm_ranker.py \
    --train-features {TRC} --val-features {VAC} --output-dir {LGBM_DIR}/lgbm_clean_full_clap
per_sub,labels=sas.batch_per_sub_rankings(queries,user_ids=user_ids,batch_context=ctx)
weights=[s['weight'] for s in sas.subs]; fused100=RRF_MODEL.fuse_per_sub(per_sub,weights,sas.k,100)
sidx=labels.index('sasrec_seq')
efpc=[]
for qi,cands in enumerate(fused100):
    rm={tid:r+1 for r,tid in enumerate(per_sub[sidx][qi])}
    efpc.append([{'sasrec_rank':rm.get(tid,10000)} for tid in cands])
esi=[{'played_tids':played[i],'turn_number':turn_numbers[i],'prior_track_count':len(played[i])} for i in range(len(queries))]
def ndcg20(ranked):
    s=0.0
    for r,g in zip(ranked,golds):
        for pos,tid in enumerate(r[:20]):
            if tid==g: s+=1.0/math.log2(pos+2); break
    return s/len(golds)
print('\n=== Stage 22 CLAP feature DEV nDCG@20 (n=%d) ===' % len(golds))
for name,sub in [('clean_full (no clap)','lgbm_clean_full'),('clean_full_clap (+clap)','lgbm_clean_full_clap')]:
    mp=f'{LGBM_DIR}/{sub}'
    if not os.path.exists(f'{mp}/booster.txt'): print('  '+name+' : (missing)'); continue
    rr=LGBM_RERANKER(ITEM_DB,['all_tracks'],CORPUS,CACHE_DIR,model_path=mp)
    rk=rr.rerank(queries,fused100,topk=20,user_ids=user_ids,goal_categories=goal_categories,
                 goal_specificities=goal_specificities,user_profiles_raw=[None]*len(queries),
                 extra_features_per_candidate=efpc,extra_session_info=esi)
    print('  '+name+' :',round(ndcg20(rk),4))
print('\n-- clean_full_clap > 0.1637 -> CLAP audio feature helps; ship. --')


## Stage 23 - CLAP audio->session RECALL channel probe (does it crack the new-artist wall?)

Stage 22 tested CLAP as a (rejected) reranker FEATURE. This tests CLAP as a NEW RECALL channel: query = mean CLAP of the session's played tracks (audio->audio, "sounds like what they've been playing") -> nearest catalog tracks. Go/no-go BEFORE building the wRRF channel: does its top-100 rescue union-missed golds, and are they NEW-ARTIST (the wall)?


In [ ]:
# Stage 23: CLAP audio->session RECALL CHANNEL probe. Requires cell 1/3 (CACHE_DIR)
# + cell 4 (queries, golds, played, item_db, recall_at, cs). No GPU, no training.
import numpy as np
from mcrs.retrieval_modules.clap_similarity import load_clap_lookup, clap_session_query

clap_lk = load_clap_lookup(CACHE_DIR)
cat_tids = list(clap_lk.keys())
cat_mat = np.stack([clap_lk[t] for t in cat_tids]).astype(np.float32)  # (Ncat, dim), L2-normed
print('[stage23] CLAP catalog matrix:', cat_mat.shape)

def _artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

TOPK = 100
K_FETCH = min(TOPK + 300, cat_mat.shape[0])  # margin to exclude played, then take TOPK
clap_cand = [[] for _ in range(len(golds))]
qvecs, qrows = [], []
for i, pl in enumerate(played):
    q = clap_session_query(pl, clap_lk)
    if q is not None:
        qvecs.append(q); qrows.append(i)
n_cold = len(golds) - len(qrows)

if qvecs:
    Q = np.stack(qvecs).astype(np.float32)               # (nq, dim)
    for s in range(0, len(Q), 256):
        sims = Q[s:s+256] @ cat_mat.T                    # (b, Ncat)
        part = np.argpartition(-sims, K_FETCH - 1, axis=1)[:, :K_FETCH]
        for r in range(sims.shape[0]):
            turn = qrows[s + r]
            played_set = set(played[turn])
            order = part[r][np.argsort(-sims[r, part[r]])]
            picks = []
            for j in order:
                t = cat_tids[j]
                if t in played_set:
                    continue
                picks.append(t)
                if len(picks) >= TOPK:
                    break
            clap_cand[turn] = picks

print('=== Stage 23 CLAP audio->session recall (dev n=%d, cold/no-history=%d) ===' % (len(golds), n_cold))
print('  CLAP-only     recall@20=%.4f  recall@100=%.4f' % (recall_at(clap_cand, 20), recall_at(clap_cand, 100)))
print('  union+SASRec  recall@100=%.4f  (cs, from cell 4)' % recall_at(cs, 100))

miss_cs = [i for i in range(len(golds)) if golds[i] not in set(cs[i][:100])]
resc = [i for i in miss_cs if golds[i] in set(clap_cand[i][:100])]
def _is_new_artist(i):
    ga = _artist_of(golds[i])
    return not (ga is not None and ga in {_artist_of(t) for t in played[i]})
resc_new = sum(1 for i in resc if _is_new_artist(i))
print('\n  union-missed @100: %d  | CLAP rescues: %d (%.1f%% of misses)' %
      (len(miss_cs), len(resc), 100.0 * len(resc) / max(1, len(miss_cs))))
if resc:
    print('    of which NEW-ARTIST: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING --')
print('  CLAP-only recall ~0 (cf. dense pre-fix 0.089) -> audio<->dialogue mismatch; channel too weak.')
print('  CLAP rescues union-missed NEW-ARTIST golds -> wall lever WORKS: build it as a wRRF sub-retriever,')
print('     union it in, re-check recall@100 + end-to-end nDCG (the real gate).')
print('  CLAP rescues ~0 -> the audio signal is already covered by SASRec (content-fused); DROP.')


## Stage 24 - preferred_musical_culture PROBE (does the user's culture tag predict the WALL golds?)

Unused signal (varies: Western/Brazilian/Hip-Hop/Metal/...). Go/no-go before wiring it as a cold-start recall prior or a culture-match rerank feature: do the union-missed (esp. NEW-ARTIST) golds' tags match the user's culture MORE than a random catalog track (LIFT)? If not, the tag is too broad to narrow the field.


In [ ]:
# Stage 24: preferred_musical_culture probe. Requires cell 4 (golds, played, user_ids,
# cs, item_db, dev). No GPU, no training.
import re, random, collections

# per-turn culture from user_profile (reuse in-memory dev + cell-4 user_ids)
u2c = {}
for s in dev:
    u2c[s.get('user_id')] = ((s.get('user_profile') or {}).get('preferred_musical_culture') or '').strip()
cultures = [u2c.get(uid, '') for uid in user_ids]
print('[stage24] turns with a culture tag: %d/%d ; distinct cultures: %d'
      % (sum(1 for c in cultures if c), len(cultures), len(set(c for c in cultures if c))))
print('  top:', ', '.join('%s(%d)' % (k, v) for k, v in
      collections.Counter(c for c in cultures if c).most_common(8)))

STOP = {'culture', 'music', 'the', 'and', 'of', 'style', 'scene'}
def _ctoks(c):
    return {w for w in re.findall(r'[a-z]+', c.lower()) if len(w) >= 3 and w not in STOP}
def _tagtext(tid):
    tags = item_db.metadata_dict.get(tid, {}).get('tag_list') or []
    return ' '.join(str(t).lower() for t in (tags if isinstance(tags, list) else [tags]))
def _match(tid, ctoks):
    if not ctoks: return False
    txt = _tagtext(tid)
    return any(t in txt for t in ctoks)

# random-catalog baseline match-rate per distinct culture (the "chance" rate)
sample = random.Random(0).sample(list(item_db.metadata_dict.keys()),
                                 min(3000, len(item_db.metadata_dict)))
sample_txt = [_tagtext(t) for t in sample]
cat_rate = {}
for c in set(c for c in cultures if c):
    ct = _ctoks(c)
    cat_rate[c] = (sum(any(t in s for t in ct) for s in sample_txt) / len(sample_txt)) if ct else 0.0

def _is_new(i):
    a = item_db.metadata_dict.get(golds[i], {}).get('artist_name')
    a = a[0] if isinstance(a, list) and a else a
    pa = {(_a[0] if isinstance(_a, list) and _a else _a) for _a in
          (item_db.metadata_dict.get(t, {}).get('artist_name') for t in played[i])}
    return not (a is not None and a in pa)

found      = [i for i in range(len(golds)) if golds[i] in set(cs[i][:100])]
missed     = [i for i in range(len(golds)) if golds[i] not in set(cs[i][:100])]
missed_new = [i for i in missed if _is_new(i)]

sc = [_match(t, _ctoks(cultures[i])) for i in range(len(golds)) if cultures[i] for t in played[i]]
print('\n  self-consistency (PLAYED tracks matching the user culture): %.1f%%'
      % (100.0 * sum(sc) / max(1, len(sc))))

print('\n=== Stage 24: GOLD-matches-culture vs random-catalog baseline ===')
for name, idxs in [('ALL golds', list(range(len(golds)))), ('FOUND golds', found),
                   ('union-MISSED golds', missed), ('MISSED NEW-ARTIST (wall)', missed_new)]:
    g = [_match(golds[i], _ctoks(cultures[i])) for i in idxs if cultures[i]]
    gm = (sum(g) / len(g)) if g else float('nan')
    bm = (sum(cat_rate[cultures[i]] for i in idxs if cultures[i]) /
          max(1, sum(1 for i in idxs if cultures[i])))
    print('  %-26s n=%-5d gold-match=%.3f  catalog-base=%.3f  LIFT=%+.3f'
          % (name, len(g), gm, bm, gm - bm))
print('\n-- READING --')
print('  LIFT on MISSED NEW-ARTIST >> 0 -> culture biases toward the wall golds: build a cold-start')
print('     culture prior / culture-match rerank feature, then gate on dev recall + nDCG.')
print('  LIFT ~0 -> culture is too broad to narrow the field (gold no more culture-matched than chance); DROP.')
print('  self-consistency low -> the label barely describes the user listening; weak signal regardless.')


## Stage 25 - image-siglip2 (album-art) -> session RECALL channel probe (the last untapped modality)

Direct clone of Stage 23, on the SigLIP2 album-art embeddings (unused anywhere). Theory: covers that look alike are stylistically alike, could reach new artists in the same visual genre. Same go/no-go: does image->session rescue union-missed NEW-ARTIST golds? Compare the numbers head-to-head with Stage 23 (CLAP).


In [ ]:
# Stage 25: image-siglip2 -> session RECALL channel probe. Requires cell 1/3 (CACHE_DIR)
# + cell 4 (golds, played, item_db, recall_at, cs). Reuses the generic mean-pool helper.
import os, pickle, numpy as np
from mcrs.retrieval_modules.clap_similarity import clap_session_query  # generic over any {tid:vec}

def _load_image_lookup(cache_dir, col='image-siglip2'):
    cache_path = os.path.join(cache_dir, 'siglip2', 'image_lookup.pkl')
    if os.path.isfile(cache_path):
        with open(cache_path, 'rb') as f:
            lk = pickle.load(f); print('[siglip2] loaded cache:', len(lk)); return lk
    import datasets as _d
    from datasets import load_dataset
    print('[siglip2] building lookup from Track-Embeddings col', col)
    ds = load_dataset('talkpl-ai/TalkPlayData-Challenge-Track-Embeddings')
    concat = _d.concatenate_datasets([ds[s] for s in ['all_tracks']])
    lk, empty = {}, 0
    for r in concat:
        emb = r.get(col)
        if not emb: empty += 1; continue
        v = np.asarray(emb, dtype=np.float32); n = float(np.linalg.norm(v))
        if n > 1e-9: lk[r['track_id']] = v / n
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)
    with open(cache_path, 'wb') as f: pickle.dump(lk, f)
    print('[siglip2] built:', len(lk), 'tracks (empty dropped=%d)' % empty); return lk

img_lk = _load_image_lookup(CACHE_DIR)
cat_tids = list(img_lk.keys())
cat_mat = np.stack([img_lk[t] for t in cat_tids]).astype(np.float32)
print('[stage25] SigLIP2 catalog matrix:', cat_mat.shape)

def _artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

TOPK = 100
K_FETCH = min(TOPK + 300, cat_mat.shape[0])
img_cand = [[] for _ in range(len(golds))]
qvecs, qrows = [], []
for i, pl in enumerate(played):
    q = clap_session_query(pl, img_lk)        # mean-pool the played tracks' IMAGE vectors
    if q is not None:
        qvecs.append(q); qrows.append(i)
n_cold = len(golds) - len(qrows)

if qvecs:
    Q = np.stack(qvecs).astype(np.float32)
    for s in range(0, len(Q), 256):
        sims = Q[s:s+256] @ cat_mat.T
        part = np.argpartition(-sims, K_FETCH - 1, axis=1)[:, :K_FETCH]
        for r in range(sims.shape[0]):
            turn = qrows[s + r]; played_set = set(played[turn])
            order = part[r][np.argsort(-sims[r, part[r]])]
            picks = []
            for j in order:
                t = cat_tids[j]
                if t in played_set: continue
                picks.append(t)
                if len(picks) >= TOPK: break
            img_cand[turn] = picks

print('=== Stage 25 image(SigLIP2)->session recall (dev n=%d, cold/no-history=%d) ===' % (len(golds), n_cold))
print('  image-only    recall@20=%.4f  recall@100=%.4f' % (recall_at(img_cand, 20), recall_at(img_cand, 100)))
print('  union+SASRec  recall@100=%.4f  (cs, from cell 4)' % recall_at(cs, 100))
miss_cs = [i for i in range(len(golds)) if golds[i] not in set(cs[i][:100])]
resc = [i for i in miss_cs if golds[i] in set(img_cand[i][:100])]
def _is_new(i):
    ga = _artist_of(golds[i])
    return not (ga is not None and ga in {_artist_of(t) for t in played[i]})
resc_new = sum(1 for i in resc if _is_new(i))
print('\n  union-missed @100: %d  | image rescues: %d (%.1f%% of misses)' %
      (len(miss_cs), len(resc), 100.0 * len(resc) / max(1, len(miss_cs))))
if resc:
    print('    of which NEW-ARTIST: %d (%.1f%%)' % (resc_new, 100.0 * resc_new / len(resc)))
print('\n-- READING (compare head-to-head with Stage 23 CLAP) --')
print('  image rescues union-missed NEW-ARTIST golds at a useful rate -> last modality has signal:')
print('     build as a wRRF channel, gate on union recall + nDCG. (Likely correlated w/ CLAP ~ genre.)')
print('  image-only recall ~0 / rescues ~0 -> album art too noisy a proxy; DROP (modality exhausted).')


## Stage 26 - CLAP recall channel union A/B (GATE A: does it lift union recall@100?)

The CLAP audio->session recall channel is now wired (use_clap_recall). Gate A: weight-swept (0.5/1.0/1.5) — does unioning it in actually raise recall@100, or does RRF noise eat it (the related-artist failure: probe 29% -> built 4%)? If it lifts recall, proceed to Gate B (rebuild LGBM features at this union -> nDCG).


In [ ]:
# Stage 26: CLAP recall channel union A/B (GATE A). Requires cell 1/3 (ITEM_DB/CORPUS/
# CACHE_DIR) + cell 4 (queries, user_ids, ctx, golds, played, item_db, recall_at).
from mcrs.retrieval_modules import load_retrieval_module

def _artist_of(tid):
    a = item_db.metadata_dict.get(tid, {}).get('artist_name')
    return (a[0] if isinstance(a, list) and a else a)

base_cfg = {'use_sasrec': True, 'w_sasrec': 1.0}
runs = [('A union+SASRec (no clap)', dict(base_cfg)),
        ('B + clap_recall w=0.5',    dict(base_cfg, use_clap_recall=True, w_clap_recall=0.5)),
        ('C + clap_recall w=1.0',    dict(base_cfg, use_clap_recall=True, w_clap_recall=1.0)),
        ('D + clap_recall w=1.5',    dict(base_cfg, use_clap_recall=True, w_clap_recall=1.5))]

print('=== Stage 26 CLAP recall channel union A/B (recall@100, dev n=%d) ===' % len(golds))
cand_by = {}
for name, cfg in runs:
    uni = load_retrieval_module('wrrf_union_v1', ITEM_DB, ['all_tracks'], CORPUS, CACHE_DIR, extra_config=cfg)
    cand = uni.batch_text_to_item_retrieval(queries, topk=100, user_ids=user_ids, batch_context=ctx)
    cand_by[name] = cand
    print('  %-28s recall@20=%.4f  recall@100=%.4f' % (name, recall_at(cand, 20), recall_at(cand, 100)))

A = cand_by['A union+SASRec (no clap)']
best = max([k for k in cand_by if not k.startswith('A ')], key=lambda k: recall_at(cand_by[k], 100))
B = cand_by[best]
miss_A = [i for i in range(len(golds)) if golds[i] not in set(A[i][:100])]
resc = [i for i in miss_A if golds[i] in set(B[i][:100])]
resc_new = sum(1 for i in resc if not (_artist_of(golds[i]) is not None
               and _artist_of(golds[i]) in {_artist_of(t) for t in played[i]}))
print('\n  best variant: %s  | delta recall@100 = %+.4f vs A' % (best, recall_at(B, 100) - recall_at(A, 100)))
print('  A-missed @100: %d | rescued by best: %d  (new-artist: %d)' % (len(miss_A), len(resc), resc_new))
print('\n-- GATE A READING --')
print('  recall@100 rises (consistently, not one fluke weight) -> realization works; do GATE B:')
print('     rebuild LGBM features at this union (cell 13 with use_clap_recall) -> retrain (14) -> e2e nDCG (16).')
print('  recall@100 ~flat or DOWN -> RRF noise ate the rescues (the related-artist pattern); DROP -> nDCG closed.')


In [ ]:
# RESULTS_JSON emitter — run after cell 49 (lgbm_relev gate, turn-stratified).
# Captures turn-1 nDCG@20 from the last reranked list (requires `reranked` and
# `golds`/`turn_numbers` from cell 4 + 12b). cat_div and lex_div are not computed
# in this retrieval-only notebook (passed as 0.0). llm_judge is None (no responder).
import sys, os
for _p in ("/content/recsys2026", os.getcwd(), os.path.dirname(os.getcwd())):
    if os.path.isdir(os.path.join(_p, "scripts")):
        sys.path.insert(0, _p); break
from scripts.emit_results import print_results_block

exp = globals().get("EXP_ID", "EXP-UNSET")
config = globals().get("CONFIG_ID", globals().get("CONFIG", 194))

# Re-run the turn-stratified report to get clean scalar values from the last reranked list.
# Requires: reranked (from cell 49), golds + turn_numbers (from cell 4).
from mcrs.eval_ndcg import ndcg_by_turn
_rep = ndcg_by_turn(reranked, golds, turn_numbers, k=20)
_ndcg_turn1   = _rep["turn1"]    # Blind proxy (cold / zero-history)
_ndcg_overall = _rep["overall"] # flat mean over all turns

print(f"[emitter] gate=turn1_cell49  n_sessions={len(golds)}")
print(f"[emitter] nDCG@20 overall={_ndcg_overall:.4f}  turn1={_ndcg_turn1:.4f}")

print_results_block(
    exp=exp,
    config=int(config),
    ndcg=_ndcg_turn1,   # turn-1 is the Blind proxy; overall also printed above
    cat_div=0.0,         # not computed in this retrieval-only gate
    lex_div=0.0,         # not computed in this retrieval-only gate
    llm_judge=None,
    n_sessions=len(golds),
    gate="turn1_cell49",
)